# ASUNDER: layout-aware cross-replica learning of common-mode fault vulnerability in diversified DMR HLS designs

This notebook reproduces the complete study end to end on a free Google Colab CPU runtime:

1. a directive-driven mini-HLS flow for eight kernels (four cryptographic, four coding/DSP),
2. LUT4/flip-flop synthesis with the open-source Yosys toolchain (YoWASP build, installed by pip),
3. bit-parallel gate-level simulation verified bit-exact against golden models,
4. quadratic placement with floorplan regions and guard bands,
5. single-spot and translated twin-spot fault-injection campaigns plus the placement-blind double bit flip,
6. the cross-replica typed-attention surrogate with spot-integral readout, GNN and gradient-boosting baselines, ablations,
7. generalisation to unseen threats and budgeted radius-robust search against random, diversity heuristic, GP-BO, REINFORCE and screening baselines,
8. the figures and statistics used in the manuscript.

`FAST = True` (default) shrinks the dataset, epochs and search so the whole notebook finishes in roughly 1.5–2.5 hours on the free CPU runtime. `FAST = False` uses the exact settings of the paper (several hours; a local multi-core machine is recommended). All random seeds are fixed.

In [ ]:
# ---- configuration -----------------------------------------------------------
import os
FAST = True
if FAST:
    os.environ.update(ASUNDER_N='80', ASUNDER_LANES='2048', ASUNDER_NTRAIN='60', ASUNDER_EPOCHS='10',
                      ASUNDER_POOL='80', ASUNDER_BUDGET='12', ASUNDER_SEARCH_SEEDS='2')
else:
    os.environ.update(ASUNDER_N='300', ASUNDER_LANES='4096', ASUNDER_NTRAIN='180', ASUNDER_EPOCHS='20',
                      ASUNDER_POOL='300', ASUNDER_BUDGET='16', ASUNDER_SEARCH_SEEDS='5')
os.environ['ASUNDER_NTRANSFER'] = '30' if FAST else '100'
SEEDS = [0] if FAST else [0, 1, 2]
for d in ['analysis', 'figs', 'data', 'results', 'models', 'search', 'graphs', 'work']:
    os.makedirs(d, exist_ok=True)
WORKERS = os.cpu_count() or 2
print('CPUs:', WORKERS, '| FAST =', FAST)

In [ ]:
!pip -q install yowasp-yosys scikit-learn scipy matplotlib lxml
import torch, sklearn, scipy, numpy; print(torch.__version__, sklearn.__version__, scipy.__version__, numpy.__version__)
!yowasp-yosys -V

### `kernels.py` — Benchmark kernels and golden models

In [ ]:
%%writefile kernels.py
"""Benchmark kernels for the ASUNDER study.

Each kernel is a loop body written as a small data-flow graph over fixed-width
words. Operand references:
  ('in', name)                  scalar input port
  ('arr', name, stride, off)    element (iter*stride+off) mod len of input array
  ('carr', name, stride, off)   element of a constant array
  ('car', var)                  loop-carried variable
  ('op', dst)                   result of a body operation
  ('const', value, width)
  ('iter',)                     loop iteration index
  ('free', fop, ref)            wiring-only transform of another reference
Every kernel also carries a bit-exact Python golden model.
"""
import random

M = lambda w: (1 << w) - 1


def _aes_sbox():
    def gmul(a, b):
        p = 0
        for _ in range(8):
            if b & 1:
                p ^= a
            hi = a & 0x80
            a = (a << 1) & 0xFF
            if hi:
                a ^= 0x1B
            b >>= 1
        return p
    inv = [0] * 256
    for a in range(1, 256):
        for b in range(1, 256):
            if gmul(a, b) == 1:
                inv[a] = b
                break
    s = []
    for a in range(256):
        x = inv[a]
        y = x
        for i in range(1, 5):
            y ^= ((x << i) | (x >> (8 - i))) & 0xFF
        s.append(y ^ 0x63)
    return s


AES_SBOX = _aes_sbox()
PRESENT_SBOX = [0xC, 0x5, 0x6, 0xB, 0x9, 0x0, 0xA, 0xD, 0x3, 0xE, 0xF, 0x8, 0x4, 0x7, 0x1, 0x2]
# 16-bit PRESENT-style bit permutation P(i) = 4*i mod 15, P(15)=15
PERM16 = [(4 * i) % 15 if i != 15 else 15 for i in range(16)]
ATAN12 = [256, 151, 80, 41, 20, 10]  # round(atan(2^-i)/(pi/4)*256), 12-bit angle scale
FIR_H = [23, 87, 87, 23]


def xtime(a):
    return ((a << 1) & 0xFF) ^ (0x1B if a & 0x80 else 0)


def crcstep(c):
    return ((c << 1) & 0xFFFF) ^ (0x1021 if c & 0x8000 else 0)


def sbox4x4(v):
    r = 0
    for n in range(4):
        r |= PRESENT_SBOX[(v >> (4 * n)) & 0xF] << (4 * n)
    return r


def perm16(v):
    r = 0
    for i in range(16):
        if (v >> i) & 1:
            r |= 1 << PERM16[i]
    return r


def sra(v, s, w):
    if v & (1 << (w - 1)):
        v = v - (1 << w)
    return (v >> s) & M(w)


def addsub(a, b, sel, w):
    return ((a - b) if sel else (a + b)) & M(w)


KERNELS = {}


def kernel(fn):
    k = fn()
    KERNELS[k['name']] = k
    return fn


@kernel
def _aes_ark():
    def golden(I):
        return {'o': [AES_SBOX[I['p'][i] ^ I['k'][i]] for i in range(4)]}
    return dict(name='aes_ark', domain='crypto', N=4,
                inputs={}, arrays={'p': (8, 4), 'k': (8, 4)}, carr={}, carried={},
                body=[('t1', 'xor', [('arr', 'p', 1, 0), ('arr', 'k', 1, 0)], 8),
                      ('t2', 'sbox8', [('op', 't1')], 8)],
                next={}, outarr={'o': (('op', 't2'), 8)}, outs={}, golden=golden)


@kernel
def _aes_mix():
    def golden(I):
        a = I['a']
        return {'o': [xtime(a[i]) ^ xtime(a[(i + 1) % 4]) ^ a[(i + 1) % 4] ^ a[(i + 2) % 4] ^ a[(i + 3) % 4]
                      for i in range(4)]}
    A = lambda o: ('arr', 'a', 1, o)
    return dict(name='aes_mix', domain='crypto', N=4,
                inputs={}, arrays={'a': (8, 4)}, carr={}, carried={},
                body=[('x1', 'xtime', [A(0)], 8), ('x2', 'xtime', [A(1)], 8),
                      ('t', 'xor', [('op', 'x2'), A(1)], 8), ('u', 'xor', [('op', 'x1'), ('op', 't')], 8),
                      ('v', 'xor', [A(2), A(3)], 8), ('w', 'xor', [('op', 'u'), ('op', 'v')], 8)],
                next={}, outarr={'o': (('op', 'w'), 8)}, outs={}, golden=golden)


@kernel
def _present():
    def golden(I):
        s = I['p0']
        for i in range(4):
            s = perm16(sbox4x4(s ^ I['k'][i]))
        return {'S': s}
    return dict(name='present', domain='crypto', N=4,
                inputs={'p0': 16}, arrays={'k': (16, 4)}, carr={}, carried={'s': (('in', 'p0'), 16)},
                body=[('t1', 'xor', [('car', 's'), ('arr', 'k', 1, 0)], 16),
                      ('t2', 'sbox4x4', [('op', 't1')], 16)],
                next={'s': ('free', 'perm16', ('op', 't2'))}, outarr={}, outs={'S': 's'}, golden=golden)


@kernel
def _speck():
    def golden(I):
        x, y = I['x0'], I['y0']
        for i in range(4):
            x = ((((x >> 7) | (x << 9)) & 0xFFFF) + y) & 0xFFFF
            x ^= I['k'][i]
            y = (((y << 2) | (y >> 14)) & 0xFFFF) ^ x
        return {'X': x, 'Y': y}
    return dict(name='speck', domain='crypto', N=4,
                inputs={'x0': 16, 'y0': 16}, arrays={'k': (16, 4)}, carr={},
                carried={'x': (('in', 'x0'), 16), 'y': (('in', 'y0'), 16)},
                body=[('b', 'add', [('free', 'ror:7', ('car', 'x')), ('car', 'y')], 16),
                      ('c', 'xor', [('op', 'b'), ('arr', 'k', 1, 0)], 16),
                      ('e', 'xor', [('free', 'rol:2', ('car', 'y')), ('op', 'c')], 16)],
                next={'x': ('op', 'c'), 'y': ('op', 'e')}, outarr={}, outs={'X': 'x', 'Y': 'y'},
                golden=golden)


@kernel
def _crc16():
    def golden(I):
        c = 0xFFFF
        for i in range(4):
            c ^= I['d'][i] << 8
            for _ in range(8):
                c = crcstep(c)
        return {'C': c}
    body = [('t0', 'xor', [('car', 'c'), ('free', 'shl8w16', ('arr', 'd', 1, 0))], 16)]
    for j in range(1, 9):
        body.append((f't{j}', 'crcstep', [('op', f't{j-1}')], 16))
    return dict(name='crc16', domain='dsp', N=4,
                inputs={}, arrays={'d': (8, 4)}, carr={}, carried={'c': (('const', 0xFFFF, 16), 16)},
                body=body, next={'c': ('op', 't8')}, outarr={}, outs={'C': 'c'}, golden=golden)


@kernel
def _fir4():
    def golden(I):
        acc = 0
        for i in range(4):
            acc = (acc + I['x'][i] * FIR_H[i]) & 0xFFFF
        return {'Y': acc}
    return dict(name='fir4', domain='dsp', N=4,
                inputs={}, arrays={'x': (8, 4)}, carr={'h': (8, FIR_H)},
                carried={'acc': (('const', 0, 16), 16)},
                body=[('p', 'mul', [('free', 'zext16', ('arr', 'x', 1, 0)),
                                    ('free', 'zext16', ('carr', 'h', 1, 0))], 16),
                      ('a', 'add', [('car', 'acc'), ('op', 'p')], 16)],
                next={'acc': ('op', 'a')}, outarr={}, outs={'Y': 'acc'}, golden=golden)


@kernel
def _dct4():
    W = 10

    def golden(I):
        x = I['x']
        out = {f'y{j}': [] for j in range(4)}
        for b in range(2):
            x0, x1, x2, x3 = x[4 * b:4 * b + 4]
            s0, s1 = (x0 + x3) & M(W), (x1 + x2) & M(W)
            d0, d1 = (x0 - x3) & M(W), (x1 - x2) & M(W)
            out['y0'].append((s0 + s1) & M(W))
            out['y2'].append((s0 - s1) & M(W))
            out['y1'].append(((d0 << 1) + d1) & M(W))
            out['y3'].append((d0 - (d1 << 1)) & M(W))
        return out
    X = lambda o: ('free', 'zext10', ('arr', 'x', 4, o))
    return dict(name='dct4', domain='dsp', N=2,
                inputs={}, arrays={'x': (8, 8)}, carr={}, carried={},
                body=[('s0', 'add', [X(0), X(3)], W), ('s1', 'add', [X(1), X(2)], W),
                      ('d0', 'sub', [X(0), X(3)], W), ('d1', 'sub', [X(1), X(2)], W),
                      ('y0', 'add', [('op', 's0'), ('op', 's1')], W),
                      ('y2', 'sub', [('op', 's0'), ('op', 's1')], W),
                      ('y1', 'add', [('free', 'shl1w10', ('op', 'd0')), ('op', 'd1')], W),
                      ('y3', 'sub', [('op', 'd0'), ('free', 'shl1w10', ('op', 'd1'))], W)],
                next={}, outarr={f'y{j}': (('op', f'y{j}'), W) for j in range(4)}, outs={},
                golden=golden)


@kernel
def _cordic():
    W = 12

    def golden(I):
        x, y, z = I['x0'], I['y0'], I['z0']
        for i in range(6):
            d = (z >> (W - 1)) & 1
            xs, ys = sra(y, i, W), sra(x, i, W)
            x, y, z = addsub(x, xs, 1 - d, W), addsub(y, ys, d, W), addsub(z, ATAN12[i], 1 - d, W)
        return {'X': x, 'Y': y, 'Z': z}
    nd = ('free', 'notmsb12', ('car', 'z'))
    dd = ('free', 'msb12', ('car', 'z'))
    return dict(name='cordic', domain='dsp', N=6,
                inputs={'x0': W, 'y0': W, 'z0': W}, arrays={}, carr={'at': (W, ATAN12)},
                carried={'x': (('in', 'x0'), W), 'y': (('in', 'y0'), W), 'z': (('in', 'z0'), W)},
                body=[('xs', 'sra', [('car', 'y'), ('iter',)], W),
                      ('ys', 'sra', [('car', 'x'), ('iter',)], W),
                      ('xn', 'addsub', [('car', 'x'), ('op', 'xs'), nd], W),
                      ('yn', 'addsub', [('car', 'y'), ('op', 'ys'), dd], W),
                      ('zn', 'addsub', [('car', 'z'), ('carr', 'at', 1, 0), nd], W)],
                next={'x': ('op', 'xn'), 'y': ('op', 'yn'), 'z': ('op', 'zn')}, outarr={},
                outs={'X': 'x', 'Y': 'y', 'Z': 'z'}, golden=golden)


def random_inputs(k, rng):
    I = {}
    for n, w in k['inputs'].items():
        I[n] = rng.getrandbits(w)
    for n, (w, L) in k['arrays'].items():
        I[n] = [rng.getrandbits(w) for _ in range(L)]
    return I


def output_layout(k):
    """Ordered list of (name, index or None, width) for the packed output vector."""
    lay = []
    for n, (_, w) in k['outarr'].items():
        for j in range(k['N']):
            lay.append((n, j, w))
    for n, var in k['outs'].items():
        lay.append((n, None, k['carried'][var][1]))
    return lay


def input_layout(k):
    lay = []
    for n, w in k['inputs'].items():
        lay.append((n, None, w))
    for n, (w, L) in k['arrays'].items():
        for j in range(L):
            lay.append((n, j, w))
    return lay


def pack(layout, D):
    v, sh = 0, 0
    for n, j, w in layout:
        x = D[n] if j is None else D[n][j]
        v |= (x & M(w)) << sh
        sh += w
    return v, sh


if __name__ == '__main__':
    rng = random.Random(0)
    for name, k in KERNELS.items():
        I = random_inputs(k, rng)
        print(name, k['golden'](I))


### `hls.py` — Directive-driven mini-HLS (unroll, list scheduling, binding, FSM emission)

In [ ]:
%%writefile hls.py
"""Minimal directive-driven HLS: unrolling, resource-constrained list scheduling,
functional-unit and register binding, FSM emission to Verilog.

Directive vector for one replica:
  u      loop unroll factor (divisor of the trip count)
  alloc  functional units per operation class (1 or 2)
  share  register binding: 'le' (left-edge sharing) or 'ded' (one register per value)
  fsm    state encoding: 'bin' or 'onehot'
  sched  list-scheduling direction: 'asap' or 'alap'
"""
import itertools
from kernels import AES_SBOX, PRESENT_SBOX, PERM16, output_layout, input_layout

FREE = {
    'ror:7': lambda e, w: (f"{{{e}[6:0],{e}[15:7]}}", 16),
    'rol:2': lambda e, w: (f"{{{e}[13:0],{e}[15:14]}}", 16),
    'shl8w16': lambda e, w: (f"{{{e}[7:0],8'd0}}", 16),
    'zext16': lambda e, w: (f"{{{16 - w}'d0,{e}}}", 16),
    'zext10': lambda e, w: (f"{{{10 - w}'d0,{e}}}", 10),
    'shl1w10': lambda e, w: (f"{{{e}[8:0],1'b0}}", 10),
    'perm16': lambda e, w: ("{" + ",".join(f"{e}[{PERM16.index(15 - k)}]" for k in range(16)) + "}", 16),
    'msb12': lambda e, w: (f"{e}[11]", 1),
    'notmsb12': lambda e, w: (f"~{e}[11]", 1),
}

FU_EXPR = {
    'add': "fa + fb", 'sub': "fa - fb", 'xor': "fa ^ fb", 'and': "fa & fb", 'or': "fa | fb",
    'mul': "fa * fb", 'sbox8': "f_sbox8(fa[7:0])", 'sbox4x4': "f_sbox4x4(fa[15:0])",
    'xtime': "{fa[6:0],1'b0} ^ (fa[7] ? 8'h1b : 8'h00)",
    'crcstep': "{fa[14:0],1'b0} ^ (fa[15] ? 16'h1021 : 16'h0000)",
    'sra': "$signed(fa) >>> fb[3:0]", 'addsub': "fc[0] ? (fa - fb) : (fa + fb)",
}
NARGS = {'add': 2, 'sub': 2, 'xor': 2, 'and': 2, 'or': 2, 'mul': 2, 'sbox8': 1, 'sbox4x4': 1,
         'xtime': 1, 'crcstep': 1, 'sra': 2, 'addsub': 3}

DIRECTIVE_SPACE = dict(u=[1, 2, 4], alloc=[1, 2], share=['le', 'ded'], fsm=['bin', 'onehot'],
                       sched=['asap', 'alap'])


def all_configs(k):
    us = [u for u in DIRECTIVE_SPACE['u'] if k['N'] % u == 0]
    out = []
    for u, a, s, f, d in itertools.product(us, DIRECTIVE_SPACE['alloc'], DIRECTIVE_SPACE['share'],
                                           DIRECTIVE_SPACE['fsm'], DIRECTIVE_SPACE['sched']):
        out.append(dict(u=u, alloc=a, share=s, fsm=f, sched=d))
    return out


def cfg_key(c):
    return f"u{c['u']}_a{c['alloc']}_{c['share']}_{c['fsm']}_{c['sched']}"


def clog2(n):
    b = 0
    while (1 << b) < n:
        b += 1
    return max(b, 1)


class Replica:
    def __init__(self, k, cfg):
        self.k, self.cfg = k, cfg
        self.U = cfg['u']
        self.NS = k['N'] // self.U
        self._unroll()
        self._schedule()
        self._bind()

    # ---------------------------------------------------------------- unroll
    def _unroll(self):
        k, U = self.k, self.U
        self.ops = []          # dict(name, base, copy, op, args, w)
        nxt_prev = None
        self.copy_next = []
        for c in range(U):
            def res(ref, c=c, nxt_prev=nxt_prev):
                t = ref[0]
                if t == 'car':
                    return ('car', ref[1]) if c == 0 else nxt_prev[ref[1]]
                if t == 'op':
                    return ('op', f"{ref[1]}#{c}")
                if t in ('arr', 'carr'):
                    return (t, ref[1], ref[2], ref[3], c)
                if t == 'iter':
                    return ('iter', c)
                if t == 'free':
                    return ('free', ref[1], res(ref[2]))
                return ref
            for dst, op, args, w in k['body']:
                self.ops.append(dict(name=f"{dst}#{c}", base=dst, copy=c, op=op,
                                     args=[res(a) for a in args], w=w))
            nxt = {v: res(r) for v, r in k['next'].items()}
            self.copy_next.append(nxt)
            self.copy_outarr = getattr(self, 'copy_outarr', [])
            self.copy_outarr.append({n: (res(r), w) for n, (r, w) in k['outarr'].items()})
            nxt_prev = nxt
        self.final_next = self.copy_next[-1] if U else {}
        self.byname = {o['name']: o for o in self.ops}

    @staticmethod
    def _op_deps(ref):
        if ref[0] == 'op':
            return [ref[1]]
        if ref[0] == 'free':
            return Replica._op_deps(ref[2])
        return []

    # -------------------------------------------------------------- schedule
    def _schedule(self):
        ops, A = self.ops, self.cfg['alloc']
        preds = {o['name']: sorted({d for a in o['args'] for d in self._op_deps(a)}) for o in ops}
        succs = {o['name']: [] for o in ops}
        for n, ps in preds.items():
            for p in ps:
                succs[p].append(n)
        order = [o['name'] for o in ops]

        def longest(adj):
            memo = {}

            def f(n):
                if n not in memo:
                    memo[n] = 1 + max([f(m) for m in adj[n]], default=0)
                return memo[n]
            return {n: f(n) for n in order}
        if self.cfg['sched'] == 'asap':
            dep, prio = preds, longest(succs)
        else:
            dep, prio = succs, longest(preds)
        step, done, t = {}, set(), 0
        while len(done) < len(order):
            t += 1
            used = {}
            ready = [n for n in order if n not in done and all(d in done and step[d] < t for d in dep[n])]
            ready.sort(key=lambda n: (-prio[n], order.index(n)))
            for n in ready:
                cls = self.byname[n]['op']
                if used.get(cls, 0) < A:
                    used[cls] = used.get(cls, 0) + 1
                    step[n] = t
                    done.add(n)
        S = t
        if self.cfg['sched'] == 'alap':
            step = {n: S - s + 1 for n, s in step.items()}
        self.state_of, self.S = step, S
        self.preds = preds

    # ------------------------------------------------------------------ bind
    def _bind(self):
        S, ops = self.S, self.ops
        UPD = S + 1
        uses = {o['name']: [] for o in ops}
        for o in ops:
            for a in o['args']:
                for d in self._op_deps(a):
                    uses[d].append(self.state_of[o['name']])
        for ref in list(self.final_next.values()):
            for d in self._op_deps(ref):
                uses[d].append(UPD)
        for oa in self.copy_outarr:
            for ref, w in oa.values():
                for d in self._op_deps(ref):
                    uses[d].append(UPD)
        self.life = {}
        for o in ops:
            sp = self.state_of[o['name']]
            last = max(uses[o['name']], default=sp + 1)
            self.life[o['name']] = (sp + 1, last)
        # register binding
        names = sorted(self.life, key=lambda n: (self.life[n][0], self.life[n][1], n))
        regs = []        # list of (end, [values])
        self.reg_of = {}
        for n in names:
            s0, s1 = self.life[n]
            placed = False
            if self.cfg['share'] == 'le':
                for r in regs:
                    if r[0] < s0:
                        r[0] = s1
                        r[1].append(n)
                        self.reg_of[n] = r[2]
                        placed = True
                        break
            if not placed:
                rid = len(regs)
                regs.append([s1, [n], rid])
                self.reg_of[n] = rid
        self.regs = [(r[2], r[1], max(self.byname[v]['w'] for v in r[1])) for r in regs]
        # functional-unit binding
        self.fu_of = {}
        self.fus = {}   # cls -> count
        for s in range(1, S + 1):
            cnt = {}
            for o in ops:
                if self.state_of[o['name']] == s:
                    cls = o['op']
                    i = cnt.get(cls, 0)
                    cnt[cls] = i + 1
                    self.fu_of[o['name']] = (cls, i)
                    self.fus[cls] = max(self.fus.get(cls, 0), i + 1)

    # ------------------------------------------------------------------ emit
    def latency(self):
        # cycle index at which done first reads 1 (IDLE is cycle 0)
        return 1 + self.NS * (self.S + 1)

    def verilog(self, modname='rep'):
        k, S, U, NS = self.k, self.S, self.U, self.NS
        UPD, DONE = S + 1, S + 2
        nst = S + 3
        L = []
        inl, outl = input_layout(k), output_layout(k)
        win = sum(w for _, _, w in inl)
        wout = sum(w for _, _, w in outl)
        L.append(f"module {modname}(input clk, input [{win - 1}:0] din, output [{wout - 1}:0] dout, output done);")
        # input buses
        off = 0
        bus = {}
        for n, w in k['inputs'].items():
            L.append(f"wire [{w - 1}:0] in_{n} = din[{off + w - 1}:{off}];")
            off += w
        for n, (w, Ln) in k['arrays'].items():
            L.append(f"wire [{w * Ln - 1}:0] bus_{n} = din[{off + w * Ln - 1}:{off}];")
            bus[n] = (w, Ln)
            off += w * Ln
        # state
        if self.cfg['fsm'] == 'bin':
            sw = clog2(nst)
            L.append(f"reg [{sw - 1}:0] state = 0;")
            for s in range(nst):
                L.append(f"wire st_{s} = (state == {sw}'d{s});")
        else:
            sw = nst - 1
            L.append(f"reg [{sw - 1}:0] state = 0;")
            L.append("wire st_0 = ~|state;")
            for s in range(1, nst):
                L.append(f"wire st_{s} = state[{s - 1}];")
        cw = clog2(NS + 1)
        if NS > 1:
            L.append(f"reg [{cw - 1}:0] cnt = 0;")
        iw = clog2(k['N']) + 1
        # registers
        for rid, vals, w in self.regs:
            L.append(f"reg [{w - 1}:0] R{rid} = 0;")
        for v, (_, w) in k['carried'].items():
            L.append(f"reg [{w - 1}:0] C_{v} = 0;")
        for n, (_, w) in k['outarr'].items():
            for j in range(k['N']):
                L.append(f"reg [{w - 1}:0] O_{n}_{j} = 0;")
        # wires for references
        self._wires = {}
        self._wl = []
        self._iw = iw

        def it_expr(c):
            if NS > 1:
                return f"(({iw}'d0 + cnt) * {U} + {c})"
            return f"{iw}'d{c}"

        def wire_of(ref):
            key = repr(ref)
            if key in self._wires:
                return self._wires[key]
            t = ref[0]
            if t == 'in':
                e, w = f"in_{ref[1]}", k['inputs'][ref[1]]
            elif t == 'car':
                e, w = f"C_{ref[1]}", k['carried'][ref[1]][1]
            elif t == 'op':
                o = self.byname[ref[1]]
                e, w = f"R{self.reg_of[ref[1]]}[{o['w'] - 1}:0]", o['w']
            elif t == 'const':
                e, w = f"{ref[2]}'d{ref[1]}", ref[2]
            elif t == 'iter':
                e, w = it_expr(ref[1]), iw
            elif t == 'arr':
                _, n, st, of, c = ref
                w, Ln = bus[n]
                idx = f"((({it_expr(c)}) * {st} + {of}) & {Ln - 1})"
                e = f"bus_{n} >> ({idx} * {w})"
            elif t == 'carr':
                _, n, st, of, c = ref
                w, vals = k['carr'][n]
                idx = f"((({it_expr(c)}) * {st} + {of}) & {len(vals) - 1 if len(vals) & (len(vals) - 1) == 0 else 7})"
                e = f"f_carr_{n}({idx})"
            elif t == 'free':
                src = wire_of(ref[2])
                e, w = FREE[ref[1]](src, self._wires_w[src])
            else:
                raise ValueError(ref)
            nm = f"w{len(self._wires)}"
            self._wl.append(f"wire [{w - 1}:0] {nm} = {e};")
            self._wires[key] = nm
            self._wires_w[nm] = w
            return nm
        self._wires_w = {}
        # functional units
        fu_lines = []
        for cls, cntu in sorted(self.fus.items()):
            ops_c = [o for o in self.ops if o['op'] == cls]
            w = max(o['w'] for o in ops_c)
            for i in range(cntu):
                mine = [o for o in ops_c if self.fu_of[o['name']] == (cls, i)]
                na = NARGS[cls]
                for ai, an in enumerate(['fa', 'fb', 'fc'][:na]):
                    terms = []
                    for o in mine:
                        terms.append((self.state_of[o['name']], wire_of(o['args'][ai])))
                    expr = terms[-1][1]
                    for s, wn in reversed(terms[:-1]):
                        expr = f"st_{s} ? {wn} : {expr}"
                    fu_lines.append(f"wire [{w - 1}:0] {cls}{i}_{an} = {expr};")
                fe = FU_EXPR[cls].replace('fa', f"{cls}{i}_fa").replace('fb', f"{cls}{i}_fb").replace('fc', f"{cls}{i}_fc")
                fu_lines.append(f"wire [{w - 1}:0] {cls}{i}_y = {fe};")
        # update-state wires
        nxt_w = {v: wire_of(r) for v, r in self.final_next.items()}
        init_w = {v: wire_of(r) for v, (r, w) in k['carried'].items()}
        out_w = [{n: wire_of(r) for n, (r, w) in oa.items()} for oa in self.copy_outarr]
        L += self._wl
        L += fu_lines
        # sequential logic
        L.append("always @(posedge clk) begin")

        def goto(s):
            if self.cfg['fsm'] == 'bin':
                return f"state <= {sw}'d{s};"
            if s == 0:
                return f"state <= {sw}'d0;"
            return f"state <= {sw}'d{1 << (s - 1)};"
        L.append(f"  if (st_0) begin {goto(1)}" + (" cnt <= 0;" if NS > 1 else "") +
                 "".join(f" C_{v} <= {init_w[v]};" for v in k['carried']) + " end")
        for s in range(1, S + 1):
            L.append(f"  if (st_{s}) {goto(s + 1)}")
        upd = [f"C_{v} <= {nxt_w[v]};" for v in k['carried']]
        for c, oa in enumerate(out_w):
            for n, wn in oa.items():
                for j in range(k['N']):
                    if (j - c) % U == 0:
                        jj = (j - c) // U
                        cond = f"cnt == {jj}" if NS > 1 else "1'b1"
                        upd.append(f"if ({cond}) O_{n}_{j} <= {wn};")
        if NS > 1:
            upd.append("cnt <= cnt + 1;")
            upd.append(f"if (cnt == {NS - 1}) {goto(DONE)} else {goto(1)}")
        else:
            upd.append(goto(DONE))
        L.append(f"  if (st_{UPD}) begin " + " ".join(upd) + " end")
        L.append(f"  if (st_{DONE}) {goto(DONE)}")
        # value registers
        for rid, vals, w in self.regs:
            branches = []
            for v in vals:
                s = self.state_of[v]
                cls, i = self.fu_of[v]
                branches.append(f"if (st_{s}) R{rid} <= {cls}{i}_y;")
            L.append("  " + " else ".join(branches))
        L.append("end")
        # outputs
        parts = []
        for n, j, w in outl:
            parts.append(f"O_{n}_{j}" if j is not None else f"C_{k['outs'][n]}")
        L.append(f"assign dout = {{{', '.join(reversed(parts))}}};")
        L.append(f"assign done = st_{DONE};")
        # functions
        if 'sbox8' in self.fus:
            L.append("function [7:0] f_sbox8(input [7:0] a); case (a)")
            L += [f"  8'd{i}: f_sbox8 = 8'd{v};" for i, v in enumerate(AES_SBOX)]
            L.append("endcase endfunction")
        if 'sbox4x4' in self.fus:
            L.append("function [3:0] f_s4(input [3:0] a); case (a)")
            L += [f"  4'd{i}: f_s4 = 4'd{v};" for i, v in enumerate(PRESENT_SBOX)]
            L.append("endcase endfunction")
            L.append("function [15:0] f_sbox4x4(input [15:0] a); f_sbox4x4 = {f_s4(a[15:12]),f_s4(a[11:8]),f_s4(a[7:4]),f_s4(a[3:0])}; endfunction")
        for n, (w, vals) in k['carr'].items():
            iwc = 4
            L.append(f"function [{w - 1}:0] f_carr_{n}(input [{iwc - 1}:0] i); case (i)")
            L += [f"  {iwc}'d{i}: f_carr_{n} = {w}'d{v};" for i, v in enumerate(vals)]
            L.append(f"  default: f_carr_{n} = {w}'d0; endcase endfunction")
        L.append("endmodule")
        return "\n".join(L)

    # ---------------------------------------------------------- timelines
    def timelines(self):
        """Live intervals of DFG values held in each register bit.

        Returns dict regname -> list of (value_key, width, t_start, t_end) in absolute
        cycles (IDLE = cycle 0). Value keys identify (base operation, iteration) and are
        therefore comparable between replicas built with different directives."""
        S, U = self.S, self.U
        tl = {}

        def key_of(ref, j):
            t = ref[0]
            if t == 'op':
                o = self.byname[ref[1]]
                return ('op', o['base'], j * U + o['copy'])
            if t == 'free':
                return ('free', ref[1], key_of(ref[2], j))
            if t == 'car':
                return ('car', ref[1], j * U)
            return ('base', repr(ref))

        def canon(key, j):
            # carried value at start of iteration i == next-expression of iteration i-1
            if key[0] == 'car':
                v, it = key[1], key[2]
                if it == 0:
                    return ('init', v)
                jj, cc = divmod(it - 1, U)
                return canon(key_of(self.copy_next[cc][v], jj), jj)
            if key[0] == 'free':
                return ('free', key[1], canon(key[2], j))
            return key
        for j in range(self.NS):
            base = 1 + j * (S + 1)
            for rid, vals, w in self.regs:
                for v in vals:
                    s0, s1 = self.life[v]
                    o = self.byname[v]
                    tl.setdefault(f"R{rid}", []).append(
                        (canon(('op', o['base'], j * U + o['copy']), j), o['w'], base + s0 - 1, base + s1 - 1))
            for v, (_, w) in self.k['carried'].items():
                key = canon(('car', v, j * U), j)
                tl.setdefault(f"C_{v}", []).append((key, w, base, base + S))
        lat = self.latency()
        for n, (ref, w) in self.k['outarr'].items():
            for j in range(self.k['N']):
                jj, c = divmod(j, U)
                key = canon(key_of(self.copy_outarr[c][n][0], jj), jj)
                tw = 1 + jj * (S + 1) + S + 1
                tl.setdefault(f"O_{n}_{j}", []).append((key, w, tw, lat + 4))
        return tl

    def role(self, regname):
        if regname == 'state':
            return 'fsm'
        if regname == 'cnt':
            return 'cnt'
        if regname.startswith('R'):
            return 'val'
        if regname.startswith('C_'):
            return 'car'
        if regname.startswith('O_'):
            return 'out'
        return 'other'


WRAPPER = """module wrap(input clk, input [{W1}:0] oa, input [{W1}:0] ob, input da, input db,
            output reg [{W1}:0] q = 0, output reg alarm = 0, output reg valid = 0);
always @(posedge clk) if (da & db & ~valid) begin valid <= 1'b1; alarm <= (oa != ob); q <= oa; end
endmodule"""


### `netlist.py` — Yosys LUT4 synthesis and bit-parallel gate-level simulator

In [ ]:
%%writefile netlist.py
"""Synthesis to a LUT4/DFF netlist with Yosys and a bit-parallel gate-level simulator."""
import json, os, re, subprocess, hashlib
import numpy as np

YOSYS = os.environ.get('YOSYS', 'yowasp-yosys')
SCRIPT = """read_verilog -sv {v}
synth -flatten -top {top}
dfflegalize -cell $_DFF_P_ 01
abc -lut 4
opt_clean -purge
setundef -zero
write_json {j}
"""


def synthesize(vtext, top, workdir, tag):
    os.makedirs(workdir, exist_ok=True)
    v = os.path.join(workdir, f"{tag}.v")
    j = os.path.join(workdir, f"{tag}.json")
    if os.path.exists(j):
        return json.load(open(j))
    open(v, 'w').write(vtext)
    ys = os.path.join(workdir, f"{tag}.ys")
    open(ys, 'w').write(SCRIPT.format(v=v, top=top, j=j))
    r = subprocess.run([YOSYS, '-q', '-s', ys], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-3000:] + r.stdout[-2000:])
    return json.load(open(j))


class Netlist:
    """Flattened netlist with LUT and FF cells over integer nets.
    Net 0 is constant 0, net 1 is constant 1."""

    def __init__(self):
        self.n = 2
        self.luts = []   # (inputs list, init int, out net, owner, name)
        self.ffs = []    # (d net, q net, init, owner, name)
        self.ports = {}  # (owner, port) -> list of nets

    def new(self):
        self.n += 1
        return self.n - 1

    def add_module(self, js, top, owner, bind=None):
        """Import a Yosys JSON module. `bind` maps port name -> list of existing nets to reuse."""
        m = js['modules'][top]
        bind = bind or {}
        local = {}

        def net(b):
            if isinstance(b, str):
                return 1 if b == '1' else 0
            if b not in local:
                local[b] = self.new()
            return local[b]
        for pn, p in m['ports'].items():
            if pn in bind:
                for b, g in zip(p['bits'], bind[pn]):
                    if isinstance(b, str):
                        continue
                    local[b] = g
        for pn, p in m['ports'].items():
            self.ports[(owner, pn)] = [net(b) for b in p['bits']]
        cand = {}
        for nn, d in m['netnames'].items():
            if d.get('hide_name', 0):
                continue
            for i, b in enumerate(d['bits']):
                if not isinstance(b, str):
                    cand.setdefault(b, []).append((nn, i))

        def rank(t):
            nn = t[0]
            for r, p in enumerate(['R', 'C_', 'O_', 'state', 'cnt', 'valid', 'alarm', 'q']):
                if nn == p or nn.startswith(p) and (p != 'R' or nn[1:].isdigit()):
                    return (r, nn)
            return (99, nn)
        names = {b: "%s[%d]" % min(v, key=rank) for b, v in cand.items()}
        for cn, c in m['cells'].items():
            t = c['type']
            if t == '$lut':
                ins = [net(b) for b in c['connections']['A']]
                init = int(c['parameters']['LUT'], 2) if isinstance(c['parameters']['LUT'], str) else int(c['parameters']['LUT'])
                y = c['connections']['Y'][0]
                self.luts.append((ins, init, net(y), owner, names.get(y, '')))
            elif t == '$_DFF_P_':
                q = c['connections']['Q'][0]
                init = 0
                a = c.get('attributes', {}).get('init')
                if a is not None:
                    init = int(a, 2) & 1 if isinstance(a, str) else int(a) & 1
                self.ffs.append((net(c['connections']['D'][0]), net(q), init, owner, names.get(q, '')))
            else:
                raise ValueError(f"unexpected cell {t}")
        return local

    # -------------------------------------------------------------- compile
    def compile(self):
        """Levelise LUTs and build vectorised evaluation tables."""
        nl = len(self.luts)
        drv = {}
        for i, (ins, init, y, o, n) in enumerate(self.luts):
            drv[y] = i
        level = [-1] * nl
        import sys
        sys.setrecursionlimit(100000)

        def lev(i):
            if level[i] >= 0:
                return level[i]
            if level[i] == -2:
                raise RuntimeError('combinational loop')
            level[i] = -2
            l = 0
            for a in self.luts[i][0]:
                if a in drv:
                    l = max(l, lev(drv[a]) + 1)
            level[i] = l
            return l
        for i in range(nl):
            lev(i)
        self.lut_level = np.array(level)
        self.levels = []
        for L in range(max(level, default=-1) + 1):
            idx = np.where(self.lut_level == L)[0]
            ins = np.zeros((len(idx), 4), dtype=np.int64)
            tab = np.zeros((len(idx), 16), dtype=np.uint64)
            outs = np.zeros(len(idx), dtype=np.int64)
            for r, i in enumerate(idx):
                a, init, y, _, _ = self.luts[i]
                w = len(a)
                aa = a + [0] * (4 - w)
                ins[r] = aa
                for m in range(16):
                    # inputs beyond WIDTH are tied to 0, so entry m maps to m & (2^w-1)
                    bit = (init >> (m & ((1 << w) - 1))) & 1
                    tab[r, m] = np.uint64(0xFFFFFFFFFFFFFFFF) if bit else np.uint64(0)
                outs[r] = y
            self.levels.append((idx, ins, tab, outs))
        self.ff_d = np.array([f[0] for f in self.ffs], dtype=np.int64)
        self.ff_q = np.array([f[1] for f in self.ffs], dtype=np.int64)
        self.ff_init = np.array([f[2] for f in self.ffs], dtype=np.uint64)

    def simulate(self, inputs_packed, n_cycles, ff_flip=None, lut_flip=None, t_inj=None,
                 probes=None):
        """Bit-parallel simulation over lanes packed in uint64 words.

        inputs_packed: dict net -> uint64 array [W] (constant over time)
        ff_flip: uint64 [n_ff, W] lanes in which each FF is upset at its lane's injection cycle
        lut_flip: uint64 [n_lut, W] lanes in which each LUT output is inverted during that cycle
        t_inj: int array [n_lanes] injection cycle per lane
        probes: list of nets; returns dict cycle -> values only at the final cycle
        Returns the final net-value matrix and per-cycle probe history."""
        W = next(iter(inputs_packed.values())).shape[0]
        V = np.zeros((self.n, W), dtype=np.uint64)
        V[1] = np.uint64(0xFFFFFFFFFFFFFFFF)
        for nnet, val in inputs_packed.items():
            V[nnet] = val
        ffs = np.repeat((self.ff_init * np.uint64(0xFFFFFFFFFFFFFFFF))[:, None], W, axis=1)
        hist = []
        tmask = None
        if t_inj is not None:
            lanes = np.arange(W * 64)
        for cyc in range(n_cycles):
            if t_inj is not None:
                sel = (t_inj == cyc)
                tmask = lanes_to_words(sel[None, :])[0] if sel.any() else None
                if tmask is not None and ff_flip is not None:
                    ffs ^= ff_flip & tmask
            V[self.ff_q] = ffs
            for L, (idx, ins, tab, outs) in enumerate(self.levels):
                a = V[ins]                     # [n,4,W]
                x0, x1, x2, x3 = a[:, 0], a[:, 1], a[:, 2], a[:, 3]
                t = tab[:, :, None]
                m = [(t[:, 2 * j + 1] & x0) | (t[:, 2 * j] & ~x0) for j in range(8)]
                m = [(m[2 * j + 1] & x1) | (m[2 * j] & ~x1) for j in range(4)]
                m = [(m[2 * j + 1] & x2) | (m[2 * j] & ~x2) for j in range(2)]
                y = (m[1] & x3) | (m[0] & ~x3)
                if tmask is not None and lut_flip is not None:
                    y ^= lut_flip[idx] & tmask
                V[outs] = y
            if probes is not None:
                hist.append(V[probes].copy())
            ffs = V[self.ff_d].copy()
        V[self.ff_q] = ffs
        return V, hist


def lanes_to_words(bits_bool):
    """bool array [..., L] -> uint64 [..., L/64] with lane l at bit (l % 64) of word l//64."""
    L = bits_bool.shape[-1]
    b = bits_bool.reshape(bits_bool.shape[:-1] + (L // 64, 64))
    w = np.zeros(b.shape[:-1], dtype=np.uint64)
    for i in range(64):
        w |= b[..., i].astype(np.uint64) << np.uint64(i)
    return w


def words_to_lanes(words):
    W = words.shape[-1]
    out = np.zeros(words.shape[:-1] + (W, 64), dtype=bool)
    for i in range(64):
        out[..., i] = ((words >> np.uint64(i)) & np.uint64(1)).astype(bool)
    return out.reshape(words.shape[:-1] + (W * 64,))


### `dmr.py` — DMR composition (replica A, replica B, comparator)

In [ ]:
%%writefile dmr.py
"""Compose a DMR design (replica A, replica B, comparator wrapper) and run campaigns."""
import random
import numpy as np
from kernels import KERNELS, random_inputs, input_layout, output_layout, pack
from hls import Replica, WRAPPER, cfg_key
from netlist import synthesize, Netlist, lanes_to_words, words_to_lanes

WORK = 'work'


def replica_json(kname, cfg):
    k = KERNELS[kname]
    r = Replica(k, cfg)
    js = synthesize(r.verilog('rep'), 'rep', f"{WORK}/{kname}", cfg_key(cfg))
    return r, js


def wrapper_json(kname):
    k = KERNELS[kname]
    wout = sum(w for _, _, w in output_layout(k))
    return synthesize(WRAPPER.format(W1=wout - 1), 'wrap', f"{WORK}/{kname}", 'wrap')


class Design:
    def __init__(self, kname, cfgA, cfgB):
        self.kname, self.k = kname, KERNELS[kname]
        self.cfgA, self.cfgB = cfgA, cfgB
        self.rA, jsA = replica_json(kname, cfgA)
        self.rB, jsB = replica_json(kname, cfgB)
        jsW = wrapper_json(kname)
        nl = Netlist()
        nl.add_module(jsA, 'rep', 'A')
        din = nl.ports[('A', 'din')]
        nl.add_module(jsB, 'rep', 'B', bind={'din': din})
        nl.add_module(jsW, 'wrap', 'W', bind={'oa': nl.ports[('A', 'dout')], 'ob': nl.ports[('B', 'dout')],
                                               'da': nl.ports[('A', 'done')], 'db': nl.ports[('B', 'done')]})
        nl.compile()
        self.nl = nl
        self.lat = max(self.rA.latency(), self.rB.latency())
        self.T = self.lat + 3

    def make_inputs(self, n_lanes, seed):
        rng = random.Random(seed)
        lay = input_layout(self.k)
        outl = output_layout(self.k)
        din = self.nl.ports[('A', 'din')]
        bits = np.zeros((len(din), n_lanes), dtype=bool)
        gold = np.zeros((sum(w for _, _, w in outl), n_lanes), dtype=bool)
        for l in range(n_lanes):
            I = random_inputs(self.k, rng)
            v, nb = pack(lay, I)
            for b in range(nb):
                bits[b, l] = (v >> b) & 1
            O = self.k['golden'](I)
            g, ng = pack(outl, O)
            for b in range(ng):
                gold[b, l] = (g >> b) & 1
        words = lanes_to_words(bits)
        return {din[b]: words[b] for b in range(len(din))}, gold

    def run(self, inputs, ff_flip=None, lut_flip=None, t_inj=None):
        nl = self.nl
        V, _ = nl.simulate(inputs, self.T, ff_flip, lut_flip, t_inj)
        q = words_to_lanes(V[nl.ports[('W', 'q')]])
        alarm = words_to_lanes(V[nl.ports[('W', 'alarm')]])[0]
        valid = words_to_lanes(V[nl.ports[('W', 'valid')]])[0]
        oa = words_to_lanes(V[nl.ports[('A', 'dout')]])
        ob = words_to_lanes(V[nl.ports[('B', 'dout')]])
        return q, alarm, valid, oa, ob

    def check(self, n_lanes=128, seed=1):
        inp, gold = self.make_inputs(n_lanes, seed)
        q, alarm, valid, oa, ob = self.run(inp)
        ok = valid.all() and (~alarm).all() and (q == gold).all()
        return ok, dict(valid=valid.mean(), alarm=alarm.mean(), qerr=(q != gold).any(0).mean(),
                        aerr=(oa != gold).any(0).mean(), berr=(ob != gold).any(0).mean())


if __name__ == '__main__':
    import sys, time
    from hls import all_configs
    names = sys.argv[1:] or list(KERNELS)
    for kn in names:
        cfgs = all_configs(KERNELS[kn])
        rng = random.Random(3)
        for trial in range(3):
            a, b = rng.choice(cfgs), rng.choice(cfgs)
            t0 = time.time()
            d = Design(kn, a, b)
            t1 = time.time()
            ok, info = d.check()
            t2 = time.time()
            print(kn, cfg_key(a), cfg_key(b), 'LUT', len(d.nl.luts), 'FF', len(d.nl.ffs), 'lat', d.rA.latency(), d.rB.latency(),
                  'OK' if ok else 'FAIL', info, f"syn {t1 - t0:.1f}s sim {t2 - t1:.2f}s", flush=True)


### `place.py` — Packing, quadratic placement, floorplan regions

In [ ]:
%%writefile place.py
"""Packing, quadratic placement with floorplan regions, and recursive-bisection legalisation."""
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spl

CAP = 8          # logic cells per tile
UTIL = 0.70      # target utilisation inside a placement region
FLOORPLANS = ['none', 'split', 'g1', 'g2']


def pack_cells(nl):
    """Return logic cells: list of dict(lut=idx or None, ff=idx or None, owner)."""
    fanout = {}
    for i, (ins, _, _, _, _) in enumerate(nl.luts):
        for a in ins:
            fanout[a] = fanout.get(a, 0) + 1
    for d, q, _, _, _ in nl.ffs:
        fanout[d] = fanout.get(d, 0) + 1
    drv = {l[2]: i for i, l in enumerate(nl.luts)}
    cells, lut_cell, ff_cell = [], {}, {}
    for j, (d, q, _, owner, _) in enumerate(nl.ffs):
        i = drv.get(d)
        if i is not None and i not in lut_cell and fanout.get(d, 0) == 1 and nl.luts[i][3] == owner:
            lut_cell[i] = len(cells)
            ff_cell[j] = len(cells)
            cells.append(dict(lut=i, ff=j, owner=owner))
        else:
            ff_cell[j] = len(cells)
            cells.append(dict(lut=None, ff=j, owner=owner))
    for i, l in enumerate(nl.luts):
        if i not in lut_cell:
            lut_cell[i] = len(cells)
            cells.append(dict(lut=i, ff=None, owner=l[3]))
    return cells, lut_cell, ff_cell


def nets_of(nl, lut_cell, ff_cell):
    """Hyperedges over logic cells (driver + sinks), excluding constants and internal LUT->FF links."""
    pins = {}
    for i, (ins, _, y, _, _) in enumerate(nl.luts):
        pins.setdefault(y, set()).add(lut_cell[i])
        for a in ins:
            if a > 1:
                pins.setdefault(a, set()).add(lut_cell[i])
    for j, (d, q, _, _, _) in enumerate(nl.ffs):
        pins.setdefault(q, set()).add(ff_cell[j])
        if d > 1:
            pins.setdefault(d, set()).add(ff_cell[j])
    return {n: sorted(s) for n, s in pins.items() if len(s) > 1}, pins


def _quadratic(n, nets, anchors, seed):
    """Solve min sum w (xi-xj)^2 with clique (<=8 pins) or star models and anchor springs."""
    rows, cols, vals = [], [], []
    nstar = 0
    extra = []
    for net in nets:
        p = len(net)
        if p <= 8:
            w = 2.0 / p
            for a in range(p):
                for b in range(a + 1, p):
                    extra.append((net[a], net[b], w))
        else:
            s = n + nstar
            nstar += 1
            for a in net:
                extra.append((a, s, 1.0))
    N = n + nstar
    deg = np.zeros(N)
    for a, b, w in extra:
        rows += [a, b]
        cols += [b, a]
        vals += [-w, -w]
        deg[a] += w
        deg[b] += w
    bx, by = np.zeros(N), np.zeros(N)
    for c, (x, y, w) in anchors.items():
        deg[c] += w
        bx[c] += w * x
        by[c] += w * y
    deg += 1e-3                                   # weak pull to origin offset below
    rows += list(range(N))
    cols += list(range(N))
    vals += list(deg)
    A = sp.csr_matrix((vals, (rows, cols)), shape=(N, N))
    rng = np.random.default_rng(seed)
    x = spl.spsolve(A.tocsc(), bx)
    y = spl.spsolve(A.tocsc(), by)
    x += rng.normal(0, 1e-3, N)
    y += rng.normal(0, 1e-3, N)
    return x[:n], y[:n]


def _bisect(ids, x, y, rect, assign):
    x0, y0, x1, y1 = rect            # tile index ranges [x0,x1) x [y0,y1)
    w, h = x1 - x0, y1 - y0
    if len(ids) == 0:
        return
    if w * h == 1:
        assign[ids] = np.array([x0, y0])
        return
    if w >= h:
        m = x0 + w // 2
        r1, r2 = (x0, y0, m, y1), (m, y0, x1, y1)
        key = x[ids]
    else:
        m = y0 + h // 2
        r1, r2 = (x0, y0, x1, m), (x0, m, x1, y1)
        key = y[ids]
    c1 = (r1[2] - r1[0]) * (r1[3] - r1[1])
    c2 = (r2[2] - r2[0]) * (r2[3] - r2[1])
    order = ids[np.argsort(key, kind='stable')]
    k = int(round(len(ids) * c1 / (c1 + c2)))
    k = min(max(k, len(ids) - c2 * CAP), c1 * CAP)
    _bisect(order[:k], x, y, r1, assign)
    _bisect(order[k:], x, y, r2, assign)


def place(nl, floorplan='none', seed=0):
    cells, lut_cell, ff_cell = pack_cells(nl)
    nets, pins = nets_of(nl, lut_cell, ff_cell)
    n = len(cells)
    owner = np.array([c['owner'] for c in cells])
    nA, nB, nW = (owner == 'A').sum(), (owner == 'B').sum(), (owner == 'W').sum()
    tiles = int(np.ceil(n / (CAP * UTIL)))
    H = max(4, int(np.ceil(np.sqrt(tiles))))
    if floorplan == 'none':
        Wd = int(np.ceil(tiles / H))
        g = 0
    else:
        g = {'split': 0, 'g1': 1, 'g2': 2}[floorplan]
        if g > 0:
            g = max(g, int(np.ceil(nW / (CAP * UTIL * H))))
        wA = int(np.ceil(nA / (CAP * UTIL * H)))
        wB = int(np.ceil(nB / (CAP * UTIL * H)))
        Wd = wA + g + wB
        if g == 0:
            wA = max(wA, int(np.ceil((nA + nW) / (CAP * UTIL * H))))
            Wd = wA + wB
    # IO anchors: shared inputs on the left edge, wrapper outputs on the right edge
    din = nl.ports[('A', 'din')]
    anchors = {}
    for i, net in enumerate(din):
        for c in pins.get(net, []):
            yv = (i + 0.5) / len(din) * H
            anchors[c] = (-1.0, yv, 0.5)
    for i, net in enumerate(nl.ports[('W', 'q')] + nl.ports[('W', 'alarm')]):
        for c in pins.get(net, []):
            anchors[c] = (Wd, (i + 0.5) / (len(nl.ports[('W', 'q')]) + 1) * H, 0.5)
    x, y = _quadratic(n, list(nets.values()), anchors, seed)
    assign = np.zeros((n, 2), dtype=np.int64)
    if floorplan == 'none':
        _bisect(np.arange(n), x, y, (0, 0, Wd, H), assign)
    else:
        if g == 0:
            _bisect(np.where(owner != 'B')[0], x, y, (0, 0, wA, H), assign)
            _bisect(np.where(owner == 'B')[0], x, y, (wA, 0, Wd, H), assign)
        else:
            _bisect(np.where(owner == 'A')[0], x, y, (0, 0, wA, H), assign)
            _bisect(np.where(owner == 'B')[0], x, y, (wA + g, 0, Wd, H), assign)
            _bisect(np.where(owner == 'W')[0], x, y, (wA, 0, wA + g, H), assign)
    # slot offsets inside tiles
    pos = assign.astype(float)
    key = assign[:, 0] * 10000 + assign[:, 1]
    order = np.argsort(key, kind='stable')
    slot = np.zeros(n, dtype=int)
    last, k = None, 0
    for i in order:
        if key[i] != last:
            last, k = key[i], 0
        slot[i] = k
        k += 1
    if slot.max() >= CAP:
        raise RuntimeError('tile overflow')
    pos[:, 0] += (slot % 2) * 0.5 - 0.25
    pos[:, 1] += (slot // 2) * 0.25 - 0.375
    hp = 0.0
    for net in nets.values():
        p = pos[net]
        hp += np.ptp(p[:, 0]) + np.ptp(p[:, 1])
    offB = {'none': Wd / 2.0, 'split': float(Wd - wB) if floorplan != 'none' else 0.0}.get(floorplan, float(Wd - wB) if floorplan != 'none' else 0.0)
    return dict(offB=offB, cells=cells, lut_cell=lut_cell, ff_cell=ff_cell, pos=pos, owner=owner,
                W=Wd, H=H, area=Wd * H, hpwl=hp, nets=nets, floorplan=floorplan)


### `fi.py` — Spot and placement-blind fault-injection campaigns

In [ ]:
%%writefile fi.py
"""Fault-injection campaigns: spatially correlated spot model and placement-blind double bit flips."""
import numpy as np
from netlist import lanes_to_words

OUTCOMES = ['S', 'D', 'C', 'H']   # silent (masked), detected, critical, hang


def twin_offset(pl):
    """Horizontal offset between the origins of the two replica regions as seen by an attacker."""
    return pl.get('offB', pl['W'] / 2.0)


def spot_masks(design, pl, n_lanes, r, m, rng):
    """Lanes x cells hit matrix for spots of radius r (tile units).
    m = 1: one spot; m = 2: twin spot, the second translated by the replica-region offset."""
    pos = pl['pos']
    cx = rng.uniform(-0.5, pl['W'] - 0.5, size=(1, n_lanes))
    cy = rng.uniform(-0.5, pl['H'] - 0.5, size=(1, n_lanes))
    if m == 3:   # two independent random spots (attacker without layout knowledge)
        cx = rng.uniform(-0.5, pl['W'] - 0.5, size=(2, n_lanes))
        cy = rng.uniform(-0.5, pl['H'] - 0.5, size=(2, n_lanes))
    elif m == 2:
        off = twin_offset(pl)
        cx0 = rng.uniform(-0.5, pl['W'] - 0.5 - off, size=n_lanes)
        cx = np.stack([cx0, cx0 + off])
        cy = np.repeat(cy, 2, 0)
    hit = np.zeros((len(pos), n_lanes), dtype=bool)
    for s in range(cx.shape[0]):
        d2 = (pos[:, 0:1] - cx[s][None]) ** 2 + (pos[:, 1:2] - cy[s][None]) ** 2
        hit |= d2 <= r * r
    return hit


def cell_to_elem(design, pl, hit):
    nl = design.nl
    ffh = np.zeros((len(nl.ffs), hit.shape[1]), dtype=bool)
    luth = np.zeros((len(nl.luts), hit.shape[1]), dtype=bool)
    for ci, c in enumerate(pl['cells']):
        if c['ff'] is not None:
            ffh[c['ff']] = hit[ci]
        if c['lut'] is not None:
            luth[c['lut']] = hit[ci]
    return ffh, luth


def classify(design, res, gold):
    q, alarm, valid, oa, ob = res
    wrongA = (oa != gold).any(0)
    wrongB = (ob != gold).any(0)
    same = (oa == ob).all(0)
    hang = ~valid
    det = valid & alarm
    crit = valid & ~alarm & (q != gold).any(0)
    sil = valid & ~alarm & ~crit
    out = np.stack([sil, det, crit, hang], 0)
    cm = wrongA & wrongB & same
    anyw = wrongA | wrongB
    return out, cm, anyw


def campaign(design, pl, r, m, n_lanes=2048, seed=0, inputs=None, model='spot'):
    rng = np.random.default_rng(seed)
    if inputs is None:
        inputs = design.make_inputs(n_lanes, seed + 7)
    inp, gold = inputs
    t_inj = rng.integers(1, design.lat, size=n_lanes)
    if model == 'spot':
        hit = spot_masks(design, pl, n_lanes, r, m, rng)
        ffh, luth = cell_to_elem(design, pl, hit)
    else:   # placement-blind double bit flip: one FF in each replica, same cycle
        nl = design.nl
        own = np.array([f[3] for f in nl.ffs])
        ia, ib = np.where(own == 'A')[0], np.where(own == 'B')[0]
        ffh = np.zeros((len(nl.ffs), n_lanes), dtype=bool)
        ffh[rng.choice(ia, n_lanes), np.arange(n_lanes)] = True
        ffh[rng.choice(ib, n_lanes), np.arange(n_lanes)] = True
        luth = np.zeros((len(nl.luts), n_lanes), dtype=bool)
    res = design.run(inp, lanes_to_words(ffh), lanes_to_words(luth), t_inj)
    out, cm, anyw = classify(design, res, gold)
    counts = out.sum(1)
    return dict(counts=counts, n=n_lanes, cm=int(cm.sum()), anyw=int(anyw.sum()),
                rates=counts / n_lanes, cmi=cm.sum() / max(anyw.sum(), 1))


### `graph.py` — Cross-replica graph (netlist, proximity, correspondence edges)

In [ ]:
%%writefile graph.py
"""Cross-replica graph construction for a placed DMR design."""
import re
import numpy as np
from scipy.spatial import cKDTree
from kernels import output_layout

ROLES = ['fsm', 'cnt', 'val', 'car', 'out', 'wrap', 'none']
R_PROX = 2.5     # tile units; covers the largest radius studied
K_PROX = 8
K_CORR = 4
F_DIM = 3 + 2 + 16 + 1 + len(ROLES) + 2 + 2 + 1 + 1


def ff_register(name, k):
    """Map a flip-flop net name to (register, bit, role)."""
    m = re.match(r'(.+)\[(\d+)\]$', name or '')
    if not m:
        return None, None, 'none'
    base, b = m.group(1), int(m.group(2))
    if base in ('state', 'done') or base.startswith('st_'):
        return 'state', b, 'fsm'
    if base == 'cnt':
        return 'cnt', b, 'cnt'
    if re.match(r'R\d+$', base):
        return base, b, 'val'
    if base.startswith('C_'):
        return base, b, 'car'
    if base.startswith('O_'):
        return base, b, 'out'
    if base == 'dout':
        off = 0
        for n, j, w in output_layout(k):
            if off <= b < off + w:
                if j is None:
                    return f"C_{k['outs'][n]}", b - off, 'car'
                return f"O_{n}_{j}", b - off, 'out'
            off += w
    if base in ('q', 'valid', 'alarm'):
        return base, b, 'wrap'
    return None, None, 'none'


def build_graph(design, pl):
    nl, k = design.nl, design.k
    cells = pl['cells']
    n = len(cells)
    own_map = {'A': 0, 'B': 1, 'W': 2}
    owner = np.array([own_map[c['owner']] for c in cells])
    pos = pl['pos'].astype(np.float32)
    X = np.zeros((n, F_DIM), dtype=np.float32)
    X[np.arange(n), owner] = 1
    fan_out = np.zeros(n)
    fan_in = np.zeros(n)
    depth = np.zeros(n)
    maxlev = max(1, int(nl.lut_level.max()) if len(nl.lut_level) else 1)
    lat = float(design.lat)
    tl = {'A': design.rA.timelines(), 'B': design.rB.timelines()}
    live = np.zeros(n)
    role_idx = np.full(n, ROLES.index('none'))
    corr_items = {}
    for ci, c in enumerate(cells):
        if c['lut'] is not None:
            ins, init, y, _, _ = nl.luts[c['lut']]
            X[ci, 3] = 1
            w = len(ins)
            for mm in range(16):
                X[ci, 5 + mm] = (init >> (mm & ((1 << w) - 1))) & 1
            X[ci, 21] = w / 4.0
            depth[ci] = nl.lut_level[c['lut']] / maxlev
        if c['ff'] is not None:
            X[ci, 4] = 1
            d, q, _, o, name = nl.ffs[c['ff']]
            reg, bit, role = ff_register(name, k)
            role_idx[ci] = ROLES.index(role)
            if o in tl and reg in tl[o]:
                for key, w, t0, t1 in tl[o][reg]:
                    if bit < w:
                        live[ci] += (t1 - t0 + 1) / lat
                        corr_items.setdefault((repr(key), bit), []).append((ci, t0, t1, o))
    X[np.arange(n), 22 + role_idx] = 1
    # netlist edges between logic cells
    src, dst = [], []
    for net, pins in pl['nets'].items():
        drv = None
        # the driver is the cell whose LUT output or FF Q is this net
        for p in pins:
            c = cells[p]
            if (c['lut'] is not None and nl.luts[c['lut']][2] == net and c['ff'] is None) or \
               (c['ff'] is not None and nl.ffs[c['ff']][1] == net):
                drv = p
                break
        if drv is None:
            continue
        for p in pins:
            if p != drv:
                src.append(drv)
                dst.append(p)
    src, dst = np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)
    np.add.at(fan_out, src, 1)
    np.add.at(fan_in, dst, 1)
    b = 22 + len(ROLES)
    X[:, b] = np.log1p(fan_out) / 4
    X[:, b + 1] = np.log1p(fan_in) / 4
    X[:, b + 2] = pos[:, 0] / pl['W']
    X[:, b + 3] = pos[:, 1] / pl['H']
    X[:, b + 4] = np.minimum(live, 4.0) / 4
    X[:, b + 5] = depth
    # proximity edges across owners
    tree = cKDTree(pos)
    dd, ii = tree.query(pos, k=min(n, 4 * K_PROX + 1), distance_upper_bound=R_PROX)
    ps, pd_, pa = [], [], []
    for i in range(n):
        cnt = 0
        for d_, j in zip(dd[i], ii[i]):
            if j >= n or j == i or owner[j] == owner[i]:
                continue
            ps.append(j); pd_.append(i); pa.append(d_)
            cnt += 1
            if cnt >= K_PROX:
                break
    # correspondence edges (A <-> B) weighted by temporal overlap of identical value bits
    off = pl.get('offB', pl['W'] / 2.0)
    pair_w = {}
    for key, lst in corr_items.items():
        As = [t for t in lst if t[3] == 'A']
        Bs = [t for t in lst if t[3] == 'B']
        for ca, a0, a1, _ in As:
            for cb, b0, b1, _ in Bs:
                ov = min(a1, b1) - max(a0, b0) + 1
                if ov > 0:
                    pair_w[(ca, cb)] = pair_w.get((ca, cb), 0.0) + ov / lat
    cs, cd, cat = [], [], []
    byA = {}
    for (ca, cb), w in pair_w.items():
        byA.setdefault(ca, []).append((w, cb))
    for ca, lst in byA.items():
        lst.sort(reverse=True)
        for w, cb in lst[:K_CORR]:
            dv = np.hypot(*(pos[cb] - pos[ca]))
            dt = np.hypot(pos[cb, 0] - pos[ca, 0] - off, pos[cb, 1] - pos[ca, 1])
            for s_, d__ in ((ca, cb), (cb, ca)):
                cs.append(s_); cd.append(d__); cat.append((w, dv, dt))
    return dict(x=X, pos=pos, owner=owner,
                e_net=np.stack([src, dst]) if len(src) else np.zeros((2, 0), np.int64),
                e_prox=np.array([ps, pd_], dtype=np.int64).reshape(2, -1),
                a_prox=np.array(pa, dtype=np.float32).reshape(-1, 1),
                e_corr=np.array([cs, cd], dtype=np.int64).reshape(2, -1),
                a_corr=np.array(cat, dtype=np.float32).reshape(-1, 3),
                W=float(pl['W']), H=float(pl['H']), offB=float(off), lat=lat)


### `model.py` — Typed-attention encoder, spot-integral readout, GNN baselines, loss

In [ ]:
%%writefile model.py
"""Cross-replica typed-attention encoder with a spot-integral readout, plus graph baselines."""
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from graph import F_DIM

TAU = 0.15
NOUT = 4


def scatter_softmax(score, index, n):
    # score [E, H]
    mx = torch.full((n, score.shape[1]), -1e30, device=score.device)
    mx = mx.scatter_reduce(0, index[:, None].expand_as(score), score, reduce='amax', include_self=True)
    ex = torch.exp(score - mx[index])
    den = torch.zeros((n, score.shape[1]), device=score.device).index_add_(0, index, ex)
    return ex / (den[index] + 1e-16)


class TypedAttention(nn.Module):
    def __init__(self, h, heads, etypes):
        super().__init__()
        self.h, self.H, self.dh = h, heads, h // heads
        self.etypes = etypes          # dict name -> attr dim
        self.q = nn.ModuleDict({t: nn.Linear(h, h, bias=False) for t in etypes})
        self.k = nn.ModuleDict({t: nn.Linear(h, h, bias=False) for t in etypes})
        self.v = nn.ModuleDict({t: nn.Linear(h, h, bias=False) for t in etypes})
        self.ab = nn.ModuleDict({t: nn.Sequential(nn.Linear(a, 16), nn.ReLU(), nn.Linear(16, heads))
                                 for t, a in etypes.items() if a > 0})
        self.o = nn.Linear(h, h)
        self.n1, self.n2 = nn.LayerNorm(h), nn.LayerNorm(h)
        self.ff = nn.Sequential(nn.Linear(h, 2 * h), nn.GELU(), nn.Linear(2 * h, h))

    def forward(self, x, edges, gates=None):
        n = x.shape[0]
        out = torch.zeros_like(x)
        for t in self.etypes:
            if t not in edges or edges[t][0].shape[1] == 0:
                continue
            e, a = edges[t]
            s, d = e[0], e[1]
            q = self.q[t](x)[d].view(-1, self.H, self.dh)
            k = self.k[t](x)[s].view(-1, self.H, self.dh)
            v = self.v[t](x)[s].view(-1, self.H, self.dh)
            sc = (q * k).sum(-1) / math.sqrt(self.dh)
            if t in self.ab:
                sc = sc + self.ab[t](a)
            al = scatter_softmax(sc, d, n)
            msg = (al[..., None] * v).reshape(-1, self.h)
            agg = torch.zeros_like(x).index_add_(0, d, msg)
            g = 1.0 if gates is None else gates[t]
            out = out + g * agg
        x = self.n1(x + self.o(out))
        return self.n2(x + self.ff(x))


def edge_attr(t, a):
    if t == 'prox':
        return torch.cat([a / 2.5, torch.exp(-a ** 2)], 1)
    if t == 'corr':
        return torch.cat([a[:, :1], a[:, 1:] / 5.0, torch.exp(-a[:, 1:] ** 2)], 1)
    return a


class Asunder(nn.Module):
    """Variants:
       use_corr / use_prox / use_pos : input ablations
       typed : separate parameters per edge type (else one merged type)
       readout : 'spot' (spot-integral) or 'pool' (global pooling with threat concatenated)"""

    def __init__(self, h=64, heads=4, layers=3, use_corr=True, use_prox=True, use_pos=True,
                 typed=True, readout='spot'):
        super().__init__()
        self.cfg = dict(use_corr=use_corr, use_prox=use_prox, use_pos=use_pos, typed=typed, readout=readout)
        et = {'net': 0, 'rnet': 0}
        if use_prox:
            et['prox'] = 2
        if use_corr:
            et['corr'] = 5
        if not typed:
            et = {'all': 6}
        self.etypes = et
        self.inp = nn.Linear(F_DIM, h)
        self.layers = nn.ModuleList([TypedAttention(h, heads, et) for _ in range(layers)])
        self.phi = nn.Linear(h, h)
        if readout == 'spot':
            self.head = nn.Sequential(nn.Linear(4 * h + 5, 2 * h), nn.GELU(), nn.Linear(2 * h, h), nn.GELU(),
                                      nn.Linear(h, NOUT + 2))
        else:
            self.head = nn.Sequential(nn.Linear(6 * h + 2, 2 * h), nn.GELU(), nn.Linear(2 * h, h), nn.GELU(),
                                      nn.Linear(h, NOUT + 2))

    def prep_edges(self, g):
        E = {}
        net = g['e_net']
        E['net'] = (net, None)
        E['rnet'] = (net.flip(0), None)
        if self.cfg['use_prox']:
            E['prox'] = (g['e_prox'], edge_attr('prox', g['a_prox']))
        if self.cfg['use_corr']:
            E['corr'] = (g['e_corr'], edge_attr('corr', g['a_corr']))
        if not self.cfg['typed']:
            es, aa = [], []
            for t, (e, a) in E.items():
                es.append(e)
                z = torch.zeros(e.shape[1], 6)
                if t == 'prox':
                    z[:, :2] = a
                elif t == 'corr':
                    z[:, 2:] = a[:, :4] if a.shape[1] >= 4 else F.pad(a, (0, 4 - a.shape[1]))
                aa.append(z)
            E = {'all': (torch.cat(es, 1), torch.cat(aa, 0))}
        return E

    def encode(self, g, gates=None):
        x = g['x']
        if not self.cfg['use_pos']:
            x = x.clone()
            x[:, 31:33] = 0
        h = self.inp(x)
        E = self.prep_edges(g)
        for l in self.layers:
            h = l(h, E, gates)
        return h

    def readout(self, g, h, threats, spots):
        """Vectorised over threats. threats: list of (r, m); spots: list of [G, 2, 2] tensors
        (single-spot threats repeat the centre). Returns P [T,4], p_cm [T], p_any [T], per-spot P [T,G,4]."""
        z = self.phi(h)
        own = g['owner']
        T = len(threats)
        if self.cfg['readout'] == 'spot':
            G = spots[0].shape[0]
            sp = torch.stack(spots)                                   # [T,G,2,2]
            d = torch.cdist(sp.reshape(-1, 2), g['pos']).view(T * G, 2, -1).min(1).values
            rv = torch.tensor([t[0] for t in threats]).repeat_interleave(G)[:, None]
            mv = torch.tensor([t[1] - 1.0 for t in threats]).repeat_interleave(G)[:, None]
            w = torch.sigmoid((rv - d) / TAU)                         # [T*G, n]
            feats, cnts = [], []
            for o in range(3):
                mk = own == o
                wo = w[:, mk]
                c = wo.sum(1, keepdim=True)
                feats.append(wo @ z[mk] / (c + 1e-3))
                cnts.append(torch.log1p(c))
            f = torch.cat(feats + [feats[0] * feats[1]] + cnts + [rv, mv], 1)
            lo = self.head(f).view(T, G, -1)
            p = torch.softmax(lo[..., :NOUT], -1)
            return p.mean(1), torch.sigmoid(lo[..., NOUT]).mean(1), torch.sigmoid(lo[..., NOUT + 1]).mean(1), p
        pools = []
        for o in range(3):
            mk = own == o
            pools += [z[mk].mean(0), z[mk].max(0).values]
        pooled = torch.cat(pools)[None].expand(T, -1)
        tv = torch.tensor([[t[0], t[1] - 1.0] for t in threats])
        lo = self.head(torch.cat([pooled, tv], 1))
        return torch.softmax(lo[:, :NOUT], 1), torch.sigmoid(lo[:, NOUT]), torch.sigmoid(lo[:, NOUT + 1]), None


class GraphBaseline(nn.Module):
    """Netlist-only GNN in the style of earlier fault-vulnerability predictors (GCN/GAT/GIN),
    global pooling and the threat parameters concatenated before the head."""

    def __init__(self, kind='gcn', h=64, layers=3):
        super().__init__()
        self.kind = kind
        keep = [3, 4] + list(range(5, 22)) + list(range(22, 29)) + [29, 30, 34]
        self.keep = keep
        self.inp = nn.Linear(len(keep), h)
        self.ws = nn.ModuleList([nn.Linear(h, h) for _ in range(layers)])
        if kind == 'gat':
            self.att = nn.ModuleList([nn.Linear(2 * h, 1) for _ in range(layers)])
        if kind == 'gin':
            self.mlps = nn.ModuleList([nn.Sequential(nn.Linear(h, h), nn.ReLU(), nn.Linear(h, h)) for _ in range(layers)])
            self.eps = nn.Parameter(torch.zeros(layers))
        self.head = nn.Sequential(nn.Linear(2 * h + 2, h), nn.ReLU(), nn.Linear(h, NOUT + 2))

    def encode(self, g, gates=None):
        x = self.inp(g['x'][:, self.keep])
        e = g['e_net']
        s = torch.cat([e[0], e[1]])
        d = torch.cat([e[1], e[0]])
        n = x.shape[0]
        deg = torch.ones(n).index_add_(0, d, torch.ones(d.shape[0]))
        for i, W in enumerate(self.ws):
            if self.kind == 'gcn':
                nrm = (deg[s] * deg[d]).rsqrt()[:, None]
                agg = torch.zeros_like(x).index_add_(0, d, nrm * x[s]) + x / deg[:, None]
                x = F.relu(W(agg))
            elif self.kind == 'gat':
                hx = W(x)
                sc = F.leaky_relu(self.att[i](torch.cat([hx[d], hx[s]], 1)), 0.2)
                al = scatter_softmax(sc, d, n)
                x = F.elu(torch.zeros_like(hx).index_add_(0, d, al * hx[s]) + hx)
            else:
                agg = torch.zeros_like(x).index_add_(0, d, x[s])
                x = self.mlps[i]((1 + self.eps[i]) * x + agg)
        return x

    def readout(self, g, h, threats, spots):
        pooled = torch.cat([h.mean(0), h.max(0).values])[None].expand(len(threats), -1)
        tv = torch.tensor([[t[0], t[1] - 1.0] for t in threats])
        lo = self.head(torch.cat([pooled, tv], 1))
        return torch.softmax(lo[:, :NOUT], 1), torch.sigmoid(lo[:, NOUT]), torch.sigmoid(lo[:, NOUT + 1]), None


def sample_spots(g, threats, G, rng):
    W, H, off = g['W'], g['H'], g['offB']
    res = []
    for r, m in threats:
        if m == 1:
            cx = rng.uniform(-0.5, W - 0.5, G)
            cy = rng.uniform(-0.5, H - 0.5, G)
            sp = np.stack([np.stack([cx, cy], 1)] * 2, 1)
        else:
            cx = rng.uniform(-0.5, W - 0.5 - off, G)
            cy = rng.uniform(-0.5, H - 0.5, G)
            sp = np.stack([np.stack([cx, cy], 1), np.stack([cx + off, cy], 1)], 1)
        res.append(torch.tensor(sp, dtype=torch.float32))
    return res


def stratified_spots(g, threats, G, rng):
    """Jittered-grid spot centres (lower variance than i.i.d. sampling)."""
    W, H, off = g['W'], g['H'], g['offB']
    s = int(np.sqrt(G))
    res = []
    for r, m in threats:
        span = (W - off) if m == 2 else W
        u = (np.arange(s)[:, None] + rng.uniform(0, 1, (s, s))) / s
        v = (np.arange(s)[None, :] + rng.uniform(0, 1, (s, s))) / s
        cx = -0.5 + u.ravel() * span
        cy = -0.5 + v.ravel() * H
        if m == 1:
            sp = np.stack([np.stack([cx, cy], 1)] * 2, 1)
        else:
            sp = np.stack([np.stack([cx, cy], 1), np.stack([cx + off, cy], 1)], 1)
        res.append(torch.tensor(sp, dtype=torch.float32))
    return res


def loss_fn(P, pcm, pany, lab, lam_c=1.0):
    """P [T,4]; lab: dict tensors counts [T,4], n [T], cm [T], anyw [T]."""
    c, n = lab['counts'], lab['n']
    eps = 1e-6
    nll = -(c * torch.log(P + eps)).sum(1) / n
    nll_cm = -(lab['cm'] * torch.log(pcm + eps) + (n - lab['cm']) * torch.log(1 - pcm + eps)) / n
    nll_an = -(lab['anyw'] * torch.log(pany + eps) + (n - lab['anyw']) * torch.log(1 - pany + eps)) / n
    rc = c[:, 2] / n
    pc = P[:, 2]
    dev = 2 * (rc * torch.log((rc + eps) / (pc + eps)) + (1 - rc) * torch.log((1 - rc + eps) / (1 - pc + eps)))
    dev = dev / (rc + 0.005)
    return (nll + 0.5 * (nll_cm + nll_an) + lam_c * dev).mean()


def collate(gs):
    """Merge several graphs into one disjoint graph for the encoder."""
    xs, pos, own, en, ep, ap, ec, ac, ptr = [], [], [], [], [], [], [], [], [0]
    for g in gs:
        o = ptr[-1]
        xs.append(g['x']); pos.append(g['pos']); own.append(g['owner'])
        en.append(g['e_net'] + o); ep.append(g['e_prox'] + o); ap.append(g['a_prox'])
        ec.append(g['e_corr'] + o); ac.append(g['a_corr'])
        ptr.append(o + g['x'].shape[0])
    return dict(x=torch.cat(xs), pos=torch.cat(pos), owner=torch.cat(own), e_net=torch.cat(en, 1),
                e_prox=torch.cat(ep, 1), a_prox=torch.cat(ap), e_corr=torch.cat(ec, 1), a_corr=torch.cat(ac),
                ptr=ptr)


### `train.py` — Training and evaluation utilities

In [ ]:
%%writefile train.py
"""Training and evaluation utilities for the surrogate models."""
import pickle, time, copy
import numpy as np
import torch
from model import Asunder, GraphBaseline, sample_spots, stratified_spots, loss_fn, collate

THREATS = [(0.6, 1), (1.2, 1), (2.0, 1), (0.6, 2), (1.2, 2), (2.0, 2)]
TKEY = lambda t: f"{t[0]}_{t[1]}"
KERNEL_ORDER = ['aes_ark', 'aes_mix', 'present', 'speck', 'crc16', 'fir4', 'dct4', 'cordic']
FOLDS = [('aes_ark', 'crc16'), ('aes_mix', 'fir4'), ('present', 'dct4'), ('speck', 'cordic')]


def to_torch(g):
    if '_t' in g:
        return g['_t']
    t = dict(x=torch.tensor(g['x']), pos=torch.tensor(g['pos']), owner=torch.tensor(g['owner']),
             e_net=torch.tensor(g['e_net']), e_prox=torch.tensor(g['e_prox']), a_prox=torch.tensor(g['a_prox']),
             e_corr=torch.tensor(g['e_corr']), a_corr=torch.tensor(g['a_corr']),
             W=g['W'], H=g['H'], offB=g['offB'])
    g['_t'] = t
    return t


def labels_of(g, threats):
    L = [g['labels'][TKEY(t)] for t in threats]
    return dict(counts=torch.tensor([l['counts'] for l in L], dtype=torch.float32),
                n=torch.tensor([l['n'] for l in L], dtype=torch.float32),
                cm=torch.tensor([l['cm'] for l in L], dtype=torch.float32),
                anyw=torch.tensor([l['anyw'] for l in L], dtype=torch.float32))


def load(kernels):
    out = []
    for k in kernels:
        out += pickle.load(open(f'graphs/{k}.pkl', 'rb'))
    return out


def make_model(spec):
    kind = spec.get('kind', 'asunder')
    if kind == 'asunder':
        return Asunder(**{k: v for k, v in spec.items() if k not in ('kind', 'name')})
    return GraphBaseline(kind)


def run_epoch(model, graphs, opt, rng, threats, G=36, bs=8):
    model.train()
    idx = rng.permutation(len(graphs))
    tot = 0.0
    for b0 in range(0, len(idx), bs):
        opt.zero_grad()
        loss = 0.0
        bidx = idx[b0:b0 + bs]
        gl = [to_torch(graphs[i]) for i in bidx]
        B = collate(gl)
        H = model.encode(B)
        for j, i in enumerate(bidx):
            g = gl[j]
            h = H[B['ptr'][j]:B['ptr'][j + 1]]
            sp = sample_spots(g, threats, G, rng)
            P, pcm, pan, _ = model.readout(g, h, threats, sp)
            loss = loss + loss_fn(P, pcm, pan, labels_of(graphs[i], threats))
        loss = loss / len(idx[b0:b0 + bs])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += float(loss.detach()) * len(bidx)
    return tot / len(idx)


@torch.no_grad()
def predict(model, graphs, threats=THREATS, G=144, seed=123):
    model.eval()
    rng = np.random.default_rng(seed)
    P = np.zeros((len(graphs), len(threats), 4))
    CM = np.zeros((len(graphs), len(threats)))
    AN = np.zeros((len(graphs), len(threats)))
    for b0 in range(0, len(graphs), 16):
        gl = [to_torch(gr) for gr in graphs[b0:b0 + 16]]
        B = collate(gl)
        H = model.encode(B)
        for j, g in enumerate(gl):
            h = H[B['ptr'][j]:B['ptr'][j + 1]]
            sp = stratified_spots(g, threats, G, rng)
            p, pcm, pan, _ = model.readout(g, h, threats, sp)
            P[b0 + j], CM[b0 + j], AN[b0 + j] = p.numpy(), pcm.numpy(), pan.numpy()
    return P, CM, AN


@torch.no_grad()
def val_loss(model, graphs, threats, seed=7):
    model.eval()
    rng = np.random.default_rng(seed)
    tot = 0.0
    for b0 in range(0, len(graphs), 16):
        grs = graphs[b0:b0 + 16]
        gl = [to_torch(gr) for gr in grs]
        B = collate(gl)
        H = model.encode(B)
        for j, g in enumerate(gl):
            h = H[B['ptr'][j]:B['ptr'][j + 1]]
            sp = stratified_spots(g, threats, 64, rng)
            P, pcm, pan, _ = model.readout(g, h, threats, sp)
            tot += float(loss_fn(P, pcm, pan, labels_of(grs[j], threats)))
    return tot / len(graphs)


def fit(spec, train_g, val_g, threats=THREATS, epochs=30, seed=0, lr=2e-3, patience=8, verbose=False):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = make_model(spec)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best, best_state, bad = 1e9, None, 0
    hist = []
    for ep in range(epochs):
        t0 = time.time()
        tl = run_epoch(model, train_g, opt, rng, threats)
        vl = val_loss(model, val_g, threats)
        sched.step()
        hist.append((tl, vl))
        if verbose:
            print(f"  ep {ep} train {tl:.4f} val {vl:.4f} {time.time() - t0:.0f}s", flush=True)
        if vl < best - 1e-4:
            best, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, hist


def truth(graphs, threats=THREATS):
    R = np.zeros((len(graphs), len(threats), 4))
    CMI = np.zeros((len(graphs), len(threats)))
    for i, g in enumerate(graphs):
        for j, t in enumerate(threats):
            l = g['labels'][TKEY(t)]
            R[i, j] = np.array(l['counts']) / l['n']
            CMI[i, j] = l['cm'] / max(l['anyw'], 1)
    return R, CMI


def split_val(graphs, frac=0.12, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(graphs))
    nv = int(len(graphs) * frac)
    return [graphs[i] for i in idx[nv:]], [graphs[i] for i in idx[:nv]]


### `exp_pred.py` — Cross-kernel prediction experiments

In [ ]:
%%writefile exp_pred.py
"""E1/E2: cross-kernel prediction experiments (grouped folds, two kernels held out per fold)."""
import sys, os, json, time, pickle
import numpy as np
import torch
from train import load, fit, predict, truth, FOLDS, KERNEL_ORDER, THREATS, split_val

SPECS = {
    'asunder': dict(kind='asunder', h=32),
    'no_corr': dict(kind='asunder', h=32, use_corr=False),
    'no_prox': dict(kind='asunder', h=32, use_prox=False),
    'no_pos': dict(kind='asunder', h=32, use_pos=False, use_prox=False),
    'untyped': dict(kind='asunder', h=32, typed=False),
    'pool_readout': dict(kind='asunder', h=32, readout='pool'),
    'gcn': dict(kind='gcn'), 'gat': dict(kind='gat'), 'gin': dict(kind='gin'),
}
N_TRAIN_PER_KERNEL = int(os.environ.get('ASUNDER_NTRAIN', 180))


def train_set(fold, seed):
    held = FOLDS[fold]
    rng = np.random.default_rng(1000 + fold)
    tr = []
    for k in KERNEL_ORDER:
        if k in held:
            continue
        gs = load([k])
        idx = rng.permutation(len(gs))[:N_TRAIN_PER_KERNEL]
        tr += [gs[i] for i in idx]
    return split_val(tr, 0.12, seed=fold)


def run(name, fold, seed, epochs=None):
    epochs = epochs or int(os.environ.get('ASUNDER_EPOCHS', 20))
    torch.set_num_threads(1)
    os.makedirs('results', exist_ok=True)
    os.makedirs('models', exist_ok=True)
    out = f'results/pred_{name}_f{fold}_s{seed}.npz'
    if os.path.exists(out):
        return
    tr, va = train_set(fold, seed)
    t0 = time.time()
    model, hist = fit(SPECS[name], tr, va, epochs=epochs, seed=seed, patience=6)
    ttrain = time.time() - t0
    torch.save(model.state_dict(), f'models/{name}_f{fold}_s{seed}.pt')
    te = load(list(FOLDS[fold]))
    t0 = time.time()
    P, CM, AN = predict(model, te)
    tinf = (time.time() - t0) / len(te)
    R, CMI = truth(te)
    ker = np.array([g['meta']['kernel'] for g in te])
    np.savez(out, P=P, CM=CM, AN=AN, R=R, CMI=CMI, ker=ker, idx=np.array([g['meta']['idx'] for g in te]),
             ttrain=ttrain, tinf=tinf, hist=np.array(hist), nparam=sum(p.numel() for p in model.parameters()))
    print(name, fold, seed, f'train {ttrain:.0f}s inf {tinf * 1000:.1f}ms', flush=True)


if __name__ == '__main__':
    jobs = [j.split(':') for j in sys.argv[1:]]
    for name, fold, seed in jobs:
        run(name, int(fold), int(seed))


### `gbm.py` — Feature-engineered gradient-boosting baseline

In [ ]:
%%writefile gbm.py
"""Feature-engineered gradient-boosting baseline with an analytical exposure feature."""
import os, sys
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from train import load, truth, FOLDS, KERNEL_ORDER, THREATS
from hls import DIRECTIVE_SPACE
from place import FLOORPLANS


def cfg_vec(c):
    v = []
    for f, opts in DIRECTIVE_SPACE.items():
        v += [float(c[f] == o) for o in opts]
    return v


def exposure(g, r, m):
    """Correspondence-weighted geometric exposure: overlap weight of matched value bits whose
    cells can be covered together by one spot (m=1) or by the translated twin spot (m=2)."""
    a = g['a_corr']
    if a.shape[0] == 0:
        return 0.0, 0.0
    w, dv, dt = a[:, 0], a[:, 1], a[:, 2]
    d = dv if m == 1 else dt
    return float((w * (d <= 2 * r)).sum() / 2), float((w * np.exp(-(d / r) ** 2)).sum() / 2)


def features(g, t, exposure_on=True):
    r, m = t
    me = g['meta']
    a, b = me['cfgA'], me['cfgB']
    ham = sum(a[f] != b[f] for f in DIRECTIVE_SPACE)
    own = g['owner']
    pa = g['a_prox'][:, 0]
    e1, e2 = exposure(g, r, m) if exposure_on else (0.0, 0.0)
    f = cfg_vec(a) + cfg_vec(b) + [float(ham), float(ham == 0)] + [float(me['fp'] == x) for x in FLOORPLANS]
    f += [me['n_lut'], me['n_ff'], me['latA'], me['latB'], me['lat'], me['W'], me['H'], me['area'], me['hpwl'],
          float((pa <= r).sum()) if exposure_on else 0.0, float((pa <= 2 * r).sum()) if exposure_on else 0.0, e1, e2, r, m]
    return f


def logit(p, eps=2e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def run(fold, seed=0, exposure_on=True):
    out = f'results/pred_gbm{"" if exposure_on else "_noexp"}_f{fold}_s{seed}.npz'
    if os.path.exists(out):
        return
    held = FOLDS[fold]
    rng = np.random.default_rng(1000 + fold)
    tr = []
    for k in KERNEL_ORDER:
        if k in held:
            continue
        gs = load([k])
        idx = rng.permutation(len(gs))[:int(os.environ.get('ASUNDER_NTRAIN', 180))]
        tr += [gs[i] for i in idx]
    te = load(list(held))
    Rtr, Ctr = truth(tr)
    Xtr = np.array([features(g, t, exposure_on) for g in tr for t in THREATS])
    Xte = np.array([features(g, t, exposure_on) for g in te for t in THREATS])
    P = np.zeros((len(te), len(THREATS), 4))
    for o in range(4):
        y = logit(Rtr[:, :, o].reshape(-1))
        mdl = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=seed).fit(Xtr, y)
        P[:, :, o] = (1 / (1 + np.exp(-mdl.predict(Xte)))).reshape(len(te), len(THREATS))
    P = P / P.sum(-1, keepdims=True)
    mdl = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=seed).fit(Xtr, Ctr.reshape(-1))
    cmi = np.clip(mdl.predict(Xte).reshape(len(te), len(THREATS)), 0, 1)
    R, CMI = truth(te)
    ker = np.array([g['meta']['kernel'] for g in te])
    np.savez(out, P=P, CM=cmi, AN=np.ones_like(cmi), R=R, CMI=CMI, ker=ker,
             idx=np.array([g['meta']['idx'] for g in te]), ttrain=0, tinf=0, hist=np.zeros((1, 2)), nparam=0)
    print('gbm', fold, flush=True)


if __name__ == '__main__':
    os.makedirs('results', exist_ok=True)
    for f in range(4):
        run(f)
        run(f, exposure_on=False)


### `gen_data.py` — Dataset generation

In [ ]:
%%writefile gen_data.py
import json, random, sys, time, os
import numpy as np
from kernels import KERNELS
from hls import all_configs, cfg_key
from dmr import Design
from place import place, FLOORPLANS
from fi import campaign

THREATS = [(0.6, 1), (1.2, 1), (2.0, 1), (0.6, 2), (1.2, 2), (2.0, 2)]
REF = dict(u=1, alloc=1, share='le', fsm='bin', sched='asap')
NL = int(os.environ.get('ASUNDER_LANES', 4096))

def evaluate(kn, a, b, fp, seed, n_lanes=NL, threats=THREATS, dbf=True):
    d = Design(kn, a, b)
    pl = place(d.nl, fp, 0)
    inp = d.make_inputs(n_lanes, seed)
    rec = dict(kernel=kn, A=cfg_key(a), B=cfg_key(b), cfgA=a, cfgB=b, fp=fp, seed=seed,
               lat=d.lat, latA=d.rA.latency(), latB=d.rB.latency(), n_lut=len(d.nl.luts), n_ff=len(d.nl.ffs),
               n_cell=len(pl['cells']), W=pl['W'], H=pl['H'], area=pl['area'], hpwl=pl['hpwl'], labels={})
    for i, (r, m) in enumerate(threats):
        res = campaign(d, pl, r, m, n_lanes, seed * 31 + i, inp)
        rec['labels'][f"{r}_{m}"] = dict(counts=res['counts'].tolist(), n=res['n'], cm=res['cm'], anyw=res['anyw'])
    if dbf:
        res = campaign(d, pl, 0, 0, n_lanes, seed * 31 + 99, inp, model='dbf')
        rec['labels']['dbf'] = dict(counts=res['counts'].tolist(), n=res['n'], cm=res['cm'], anyw=res['anyw'])
    return rec

def designs_for(kn, n=None, seed=0):
    n = n or int(os.environ.get('ASUNDER_N', 300))
    cf = all_configs(KERNELS[kn]); rng = random.Random(seed + sum(map(ord, kn)))
    out = [(REF, REF, fp) for fp in FLOORPLANS]
    seen = set((cfg_key(a), cfg_key(b), fp) for a, b, fp in out)
    while len(out) < n:
        a = rng.choice(cf)
        b = a if rng.random() < 0.2 else rng.choice(cf)
        fp = rng.choice(FLOORPLANS)
        key = (cfg_key(a), cfg_key(b), fp)
        if key in seen: continue
        seen.add(key); out.append((a, b, fp))
    return out

if __name__ == '__main__':
    kns = sys.argv[1].split(',')
    os.makedirs('data', exist_ok=True)
    for kn in kns:
        fn = f'data/{kn}.jsonl'
        done = sum(1 for _ in open(fn)) if os.path.exists(fn) else 0
        ds = designs_for(kn)
        t0 = time.time()
        with open(fn, 'a') as f:
            for i, (a, b, fp) in enumerate(ds):
                if i < done: continue
                rec = evaluate(kn, a, b, fp, seed=1000 + i)
                rec['idx'] = i
                f.write(json.dumps(rec) + '\n'); f.flush()
                if i % 25 == 0: print(kn, i, f"{time.time()-t0:.0f}s", flush=True)


### `cache_graphs.py` — Graph caching

In [ ]:
%%writefile cache_graphs.py
import json, pickle, sys, os
import numpy as np
from dmr import Design
from place import place
from graph import build_graph
from hls import cfg_key
THREATS = ['0.6_1','1.2_1','2.0_1','0.6_2','1.2_2','2.0_2']
def build(rec):
    d = Design(rec['kernel'], rec['cfgA'], rec['cfgB'])
    pl = place(d.nl, rec['fp'], 0)
    g = build_graph(d, pl)
    g['labels'] = rec['labels']; g['meta'] = {k: rec[k] for k in rec if k not in ('labels',)}
    return g
if __name__ == '__main__':
    os.makedirs('graphs', exist_ok=True)
    for kn in sys.argv[1].split(','):
        recs = [json.loads(l) for l in open(f'data/{kn}.jsonl')]
        gs = [build(r) for r in recs]
        pickle.dump(gs, open(f'graphs/{kn}.pkl', 'wb'))
        print(kn, len(gs), flush=True)


### `gen_transfer.py` — Labels for threats never used in training

In [ ]:
%%writefile gen_transfer.py
"""Labels for threats never used in training: interpolated/extrapolated radius and a random double spot."""
import json, sys, os
from dmr import Design
from place import place
from fi import campaign
from train import KERNEL_ORDER
NEW = [(0.9, 1), (0.9, 2), (2.6, 1), (2.6, 2), (1.2, 3)]
N, LANES = int(os.environ.get('ASUNDER_NTRANSFER', 100)), int(os.environ.get('ASUNDER_LANES_T', 2048))
if __name__ == '__main__':
    for kn in sys.argv[1].split(','):
        out = {}
        for l in open(f'data/{kn}.jsonl'):
            r = json.loads(l)
            if r['idx'] >= N:
                continue
            d = Design(kn, r['cfgA'], r['cfgB']); pl = place(d.nl, r['fp'], 0)
            inp = d.make_inputs(LANES, 77000 + r['idx'])
            lab = {}
            for i, (rr, m) in enumerate(NEW):
                res = campaign(d, pl, rr, m, LANES, 88000 + 10 * r['idx'] + i, inp)
                lab[f'{rr}_{m}'] = dict(counts=res['counts'].tolist(), n=res['n'], cm=res['cm'], anyw=res['anyw'])
            out[r['idx']] = lab
        json.dump(out, open(f'data/transfer_{kn}.json', 'w'))
        print(kn, len(out), flush=True)


### `search.py` — Budgeted radius-robust search

In [ ]:
%%writefile search.py
"""E5: budgeted search for threat-robust DMR designs (directives of both replicas + floorplan)."""
import os, sys, json, time, random, pickle
import numpy as np
import torch
from kernels import KERNELS
from hls import all_configs, cfg_key, DIRECTIVE_SPACE
from dmr import Design
from place import place, FLOORPLANS
from fi import campaign
from graph import build_graph
from train import FOLDS, KERNEL_ORDER, load, truth, to_torch, make_model, THREATS
from model import stratified_spots, collate
from exp_pred import SPECS

REF = dict(u=1, alloc=1, share='le', fsm='bin', sched='asap')
OBJ_T = THREATS
AREA_CAP, LAT_CAP = 1.30, 1.50
B_FI = int(os.environ.get('ASUNDER_BUDGET', 16))
POOL = int(os.environ.get('ASUNDER_POOL', 300))
EPS = 1e-4


def fold_of(kn):
    return [i for i, f in enumerate(FOLDS) if kn in f][0]


def build(kn, a, b, fp):
    d = Design(kn, a, b)
    pl = place(d.nl, fp, 0)
    return d, pl


def ref_costs(kn):
    d, pl = build(kn, REF, REF, 'none')
    return pl['area'], d.lat


def feasible(d, pl, ref):
    return pl['area'] <= AREA_CAP * ref[0] and d.lat <= LAT_CAP * ref[1]


def make_pool(kn, seed=0):
    fn = f'search/pool_{kn}.pkl'
    if os.path.exists(fn):
        return pickle.load(open(fn, 'rb'))
    cf = all_configs(KERNELS[kn])
    ref = ref_costs(kn)
    rng = random.Random(seed + sum(map(ord, kn)))
    pool, seen = [], set()
    tries = 0
    while len(pool) < POOL and tries < 20 * POOL:
        tries += 1
        a = rng.choice(cf)
        b = a if rng.random() < 0.15 else rng.choice(cf)
        fp = rng.choice(FLOORPLANS)
        key = (cfg_key(a), cfg_key(b), fp)
        if key in seen:
            continue
        seen.add(key)
        d, pl = build(kn, a, b, fp)
        if not feasible(d, pl, ref):
            continue
        g = build_graph(d, pl)
        g['meta'] = dict(kernel=kn, cfgA=a, cfgB=b, fp=fp, area=pl['area'], lat=d.lat, n_lut=len(d.nl.luts),
                         n_ff=len(d.nl.ffs), latA=d.rA.latency(), latB=d.rB.latency(), W=pl['W'], H=pl['H'],
                         hpwl=pl['hpwl'])
        pool.append(g)
    os.makedirs('search', exist_ok=True)
    pickle.dump((pool, ref), open(fn, 'wb'))
    return pool, ref


def true_J(kn, meta, n_lanes=4096, seed=0):
    d, pl = build(kn, meta['cfgA'], meta['cfgB'], meta['fp'])
    seed = int(seed)
    inp = d.make_inputs(n_lanes, 50000 + seed)
    c = [campaign(d, pl, r, m, n_lanes, 70000 + seed * 7 + i, inp)['rates'][2] for i, (r, m) in enumerate(OBJ_T)]
    return max(c), c


# ------------------------------------------------------------------ surrogates
@torch.no_grad()
def ensemble_scores(kn, graphs, seeds=(0, 1, 2)):
    f = fold_of(kn)
    preds = []
    for s in seeds:
        m = make_model(SPECS['asunder'])
        m.load_state_dict(torch.load(f'models/asunder_f{f}_s{s}.pt'))
        m.eval()
        rng = np.random.default_rng(11)
        out = np.zeros((len(graphs), len(OBJ_T)))
        for b0 in range(0, len(graphs), 16):
            gl = [to_torch(g) for g in graphs[b0:b0 + 16]]
            Bt = collate(gl)
            H = m.encode(Bt)
            for j, g in enumerate(gl):
                h = H[Bt['ptr'][j]:Bt['ptr'][j + 1]]
                P, _, _, _ = m.readout(g, h, OBJ_T, stratified_spots(g, OBJ_T, 100, rng))
                out[b0 + j] = P[:, 2].numpy()
        preds.append(out.max(1))
    return np.array(preds)            # [S, n]


def gbm_scores(kn, graphs):
    from gbm import features, logit
    from sklearn.ensemble import HistGradientBoostingRegressor
    f = fold_of(kn)
    tr = []
    rng = np.random.default_rng(1000 + f)
    for k in KERNEL_ORDER:
        if k in FOLDS[f]:
            continue
        gs = load([k])
        idx = rng.permutation(len(gs))[:int(os.environ.get('ASUNDER_NTRAIN', 180))]
        tr += [gs[i] for i in idx]
    R, _ = truth(tr, OBJ_T)
    X = np.array([features(g, t) for g in tr for t in OBJ_T])
    mdl = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=0).fit(X, logit(R[:, :, 2].reshape(-1)))
    Xp = np.array([features(g, t) for g in graphs for t in OBJ_T])
    p = 1 / (1 + np.exp(-mdl.predict(Xp)))
    return p.reshape(len(graphs), len(OBJ_T)).max(1)


# ---------------------------------------------------------------------- encodings
def encode(meta):
    v = []
    for c in (meta['cfgA'], meta['cfgB']):
        for f, opts in DIRECTIVE_SPACE.items():
            v += [float(c[f] == o) for o in opts]
    v += [float(meta['fp'] == x) for x in FLOORPLANS]
    return np.array(v)


# -------------------------------------------------------------------------- methods
def m_random(kn, pool, rng, Jcache):
    idx = rng.choice(len(pool), B_FI, replace=False)
    return list(idx)


def m_diversity(kn, pool, rng, Jcache):
    """Design-diversity rule of thumb: maximum directive Hamming distance, widest isolation."""
    order = {'g2': 3, 'g1': 2, 'split': 1, 'none': 0}
    sc = []
    for i, g in enumerate(pool):
        a, b = g['meta']['cfgA'], g['meta']['cfgB']
        ham = sum(a[f] != b[f] for f in DIRECTIVE_SPACE)
        sc.append((ham, order[g['meta']['fp']], rng.random(), i))
    sc.sort(reverse=True)
    return [sc[0][3]]


def m_screen(scores):
    def f(kn, pool, rng, Jcache):
        return list(np.argsort(scores)[:B_FI])
    return f


def m_gpbo(kn, pool, rng, Jcache, n_init=4):
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
    from scipy.stats import norm
    X = np.array([encode(g['meta']) for g in pool])
    chosen = list(rng.choice(len(pool), n_init, replace=False))
    while len(chosen) < B_FI:
        y = np.log(np.array([Jcache(i) for i in chosen]) + EPS)
        gp = GaussianProcessRegressor(ConstantKernel(1.0) * Matern(length_scale=2.0, nu=2.5) + WhiteKernel(0.1),
                                      normalize_y=True, random_state=0).fit(X[chosen], y)
        mu, sd = gp.predict(X, return_std=True)
        best = y.min()
        z = (best - mu) / (sd + 1e-9)
        ei = (best - mu) * norm.cdf(z) + sd * norm.pdf(z)
        ei[chosen] = -1
        chosen.append(int(np.argmax(ei)))
    return chosen


def m_reinforce(kn, pool, rng, Jcache, batch=4, lr=0.5):
    """Factorised categorical policy over the 11 design factors, restricted to the feasible pool:
    each sample is the pool member with the highest policy log-probability among 32 random draws."""
    X = np.array([encode(g['meta']) for g in pool])
    logits = np.zeros(X.shape[1])
    groups = []
    k = 0
    for rep in range(2):
        for f, opts in DIRECTIVE_SPACE.items():
            groups.append(list(range(k, k + len(opts))))
            k += len(opts)
    groups.append(list(range(k, k + len(FLOORPLANS))))
    chosen, base = [], None
    while len(chosen) < B_FI:
        lp = np.zeros(len(pool))
        for gidx in groups:
            l = logits[gidx] - np.log(np.exp(logits[gidx]).sum())
            lp += X[:, gidx] @ l
        batch_idx = []
        for _ in range(batch):
            cand = rng.choice(len(pool), min(32, len(pool)), replace=False)
            p = np.exp(lp[cand] - lp[cand].max())
            p /= p.sum()
            i = int(rng.choice(cand, p=p))
            if i not in chosen and i not in batch_idx:
                batch_idx.append(i)
        batch_idx = batch_idx[:B_FI - len(chosen)]
        if not batch_idx:
            continue
        rew = -np.log(np.array([Jcache(i) for i in batch_idx]) + EPS)
        base = rew.mean() if base is None else 0.7 * base + 0.3 * rew.mean()
        for i, rw in zip(batch_idx, rew):
            for gidx in groups:
                pr = np.exp(logits[gidx]) / np.exp(logits[gidx]).sum()
                logits[gidx] += lr * (rw - base) * (X[i, gidx] - pr)
        chosen += batch_idx
    return chosen


def m_asunder(ens, kappa=1.0, lam=2.0):
    """Ensemble-guided acquisition with few-shot log-linear recalibration on the target kernel."""
    lmu = np.log(ens + EPS).mean(0)
    lsd = np.log(ens + EPS).std(0)

    def f(kn, pool, rng, Jcache):
        chosen = [int(np.argmin(lmu))]
        while len(chosen) < B_FI:
            x = lmu[chosen]
            y = np.log(np.array([Jcache(i) for i in chosen]) + EPS)
            # ridge fit of y = a + b x shrunk toward (0, 1)
            A = np.stack([np.ones_like(x), x], 1)
            P0 = np.array([0.0, 1.0])
            reg = lam * np.eye(2)
            th = np.linalg.solve(A.T @ A + reg, A.T @ y + reg @ P0)
            res = y - A @ th
            s = np.sqrt((res ** 2).sum() / max(len(y) - 1, 1)) if len(y) > 2 else 0.5
            pred = th[0] + th[1] * lmu
            lcb = pred - kappa * (abs(th[1]) * lsd + s)
            lcb[chosen] = np.inf
            chosen.append(int(np.argmin(lcb)))
        return chosen
    return f


def run_kernel(kn, seeds=None):
    seeds = seeds or range(int(os.environ.get('ASUNDER_SEARCH_SEEDS', 5)))
    os.makedirs('search', exist_ok=True)
    out = f'search/res_{kn}.json'
    if os.path.exists(out):
        return
    t0 = time.time()
    pool, ref = make_pool(kn)
    tpool = time.time() - t0
    t0 = time.time()
    ens = ensemble_scores(kn, pool)
    tens = (time.time() - t0) / (len(pool) * ens.shape[0])
    gb = gbm_scores(kn, pool)
    ref_meta = dict(cfgA=REF, cfgB=REF, fp='none')
    J_ref, _ = true_J(kn, ref_meta, 16384, 999)
    cache = {}

    def Jc(i):
        if i not in cache:
            cache[int(i)] = true_J(kn, pool[i]["meta"], 4096, int(i))[0]
            return cache[int(i)]
        return cache[i]
    t0 = time.time()
    for i in range(len(pool)):
        Jc(i)
    t_fi = (time.time() - t0) / len(pool)
    res_tfi = t_fi
    methods = {'random': m_random, 'diversity': m_diversity, 'gp_bo': m_gpbo, 'reinforce': m_reinforce,
               'gbm_screen': m_screen(gb), 'asunder_screen': m_screen(ens.mean(0)), 'asunder': m_asunder(ens)}
    res = dict(t_fi=res_tfi, kernel=kn, ref_area=ref[0], ref_lat=ref[1], J_ref=J_ref, t_pool=tpool, t_surrogate=tens,
               pool_size=len(pool), runs=[])
    final_cache = {}
    for s in seeds:
        for name, meth in methods.items():
            rng = np.random.default_rng(100 * s + sum(map(ord, name)))
            chosen = meth(kn, pool, rng, Jc)
            obs = [Jc(i) for i in chosen]
            best = chosen[int(np.argmin(obs))]
            curve = list(np.minimum.accumulate(obs))
            if best not in final_cache:
                final_cache[best] = true_J(kn, pool[best]['meta'], 16384, 777)
            Jf, comps = final_cache[best]
            me = pool[best]['meta']
            res['runs'].append(dict(seed=s, method=name, chosen=[int(c) for c in chosen], obs=obs, curve=curve,
                                    best=int(best), J_final=Jf, C_single=comps[0], C_twin=comps[1],
                                    area=me['area'], lat=me['lat'], fp=me['fp'], A=cfg_key(me['cfgA']),
                                    B=cfg_key(me['cfgB'])))
        print(kn, 'seed', s, {r['method']: round(r['J_final'], 5) for r in res['runs'] if r['seed'] == s}, flush=True)
    # pool oracle (every pool member was evaluated once with 4096 faults)
    res['oracle_seen'] = min(cache.values())
    res['n_evaluated'] = len(cache)
    res['ens_mu'] = ens.mean(0).tolist()
    res['ens_sd'] = ens.std(0).tolist()
    res['gbm'] = gb.tolist()
    res['J_cache'] = {int(k): v for k, v in cache.items()}
    json.dump(res, open(out, 'w'))


if __name__ == '__main__':
    for kn in sys.argv[1].split(','):
        run_kernel(kn)


### `search_gp.py` — Post hoc ASUNDER-GP search variant (reported as a negative result)

In [ ]:
%%writefile search_gp.py
"""ASUNDER-GP: Gaussian-process search whose prior mean is the recalibrated ensemble surrogate.
Uses the already measured pool objectives stored by search.py, so no new pool campaigns are needed;
the returned design is re-measured with the same 16,384-run protocol as every other method."""
import sys, json, pickle, os
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
import warnings; warnings.filterwarnings('ignore')
import search
from hls import cfg_key
EPS = search.EPS

def m_asunder_gp(ens, lam=2.0):
    lmu = np.log(ens + EPS).mean(0); lsd = np.log(ens + EPS).std(0)
    def f(kn, pool, rng, Jcache):
        X = np.array([search.encode(g['meta']) for g in pool])
        chosen = [int(np.argmin(lmu))]
        while len(chosen) < search.B_FI:
            x = lmu[chosen]; y = np.log(np.array([Jcache(i) for i in chosen]) + EPS)
            A = np.stack([np.ones_like(x), x], 1); P0 = np.array([0.0, 1.0]); reg = lam * np.eye(2)
            th = np.linalg.solve(A.T @ A + reg, A.T @ y + reg @ P0)
            prior = th[0] + th[1] * lmu
            res = y - prior[chosen]
            gp = GaussianProcessRegressor(ConstantKernel(1.0) * Matern(length_scale=2.0, nu=2.5) + WhiteKernel(0.1),
                                          normalize_y=False, random_state=0).fit(X[chosen], res)
            rm, rs = gp.predict(X, return_std=True)
            mu = prior + rm
            sd = np.sqrt(rs ** 2 + (th[1] * lsd) ** 2)
            best = y.min(); z = (best - mu) / (sd + 1e-9)
            ei = (best - mu) * norm.cdf(z) + sd * norm.pdf(z)
            ei[chosen] = -1
            chosen.append(int(np.argmax(ei)))
        return chosen
    return f

def run(kn):
    fn = f'search/res_{kn}.json'
    res = json.load(open(fn))
    if any(r['method'] == 'asunder_gp' for r in res['runs']):
        return
    pool, ref = pickle.load(open(f'search/pool_{kn}.pkl', 'rb'))
    ens = search.ensemble_scores(kn, pool)
    cache = {int(k): v for k, v in res['J_cache'].items()}
    Jc = lambda i: cache[int(i)]
    meth = m_asunder_gp(ens)
    final = {}
    for s in range(int(os.environ.get('ASUNDER_SEARCH_SEEDS', 5))):
        rng = np.random.default_rng(100 * s + 7)
        chosen = meth(kn, pool, rng, Jc)
        obs = [Jc(i) for i in chosen]; best = chosen[int(np.argmin(obs))]
        if best not in final:
            final[best] = search.true_J(kn, pool[best]['meta'], 16384, 777)
        Jf, comps = final[best]; me = pool[best]['meta']
        res['runs'].append(dict(seed=s, method='asunder_gp', chosen=[int(c) for c in chosen], obs=obs,
                                curve=list(np.minimum.accumulate(obs)), best=int(best), J_final=Jf, C_single=comps[0], C_twin=comps[1],
                                area=me['area'], lat=me['lat'], fp=me['fp'], A=cfg_key(me['cfgA']), B=cfg_key(me['cfgB'])))
    json.dump(res, open(fn, 'w'))
    print(kn, 'asunder_gp', [round(100 * r['J_final'], 3) for r in res['runs'] if r['method'] == 'asunder_gp'], flush=True)


if __name__ == '__main__':
    for kn in sys.argv[1].split(','):
        run(kn)


### `e0.py` — Dataset analysis

In [ ]:
%%writefile e0.py
import json, numpy as np
from scipy.stats import spearmanr
from train import KERNEL_ORDER
T=['0.6_1','1.2_1','2.0_1','0.6_2','1.2_2','2.0_2']
rows=[]
for kn in KERNEL_ORDER:
    for l in open(f'data/{kn}.jsonl'):
        r=json.loads(l); rows.append(r)
def rate(r,t,o=2): L=r['labels'][t]; return L['counts'][o]/L['n']
def ham(r): return sum(r['cfgA'][f]!=r['cfgB'][f] for f in r['cfgA'])
out={}
# 1. rank agreement DBF vs spatial
rho={}
for kn in KERNEL_ORDER:
    R=[r for r in rows if r['kernel']==kn]
    d=np.array([rate(r,'dbf') for r in R])
    rho[kn]={t: spearmanr(d,[rate(r,t) for r in R]).correlation for t in T}
    rho[kn]['J']=spearmanr(d,[max(rate(r,t) for t in T) for r in R]).correlation
out['rho_dbf']=rho
print('Spearman DBF-CER vs spatial CER:')
for kn in KERNEL_ORDER: print(kn, {k: round(v,2) for k,v in rho[kn].items()})
print('mean', {t: round(np.mean([rho[k][t] for k in KERNEL_ORDER]),2) for t in T+['J']})
# 2. floorplan x identical/diverse
print('\nmean CER (%) by floorplan and diversity (all kernels)')
tab={}
for fp in ['none','split','g1','g2']:
    for div in ['ident','diverse']:
        S=[r for r in rows if r['fp']==fp and ((ham(r)==0)==(div=='ident'))]
        tab[f'{fp}_{div}']={t: 100*np.mean([rate(r,t) for r in S]) for t in T}
        tab[f'{fp}_{div}']['n']=len(S)
        tab[f'{fp}_{div}']['J']=100*np.mean([max(rate(r,t) for t in T) for r in S])
        tab[f'{fp}_{div}']['cmi12_2']=np.mean([r['labels']['1.2_2']['cm']/max(r['labels']['1.2_2']['anyw'],1) for r in S])
        print(fp,div,len(S),{k: round(v,3) for k,v in tab[f'{fp}_{div}'].items()})
out['fp_div']=tab
# 3. per-kernel stats
print('\nper kernel: mean/median/min J, REF J (none), LUT range')
pk={}
for kn in KERNEL_ORDER:
    R=[r for r in rows if r['kernel']==kn]
    J=np.array([max(rate(r,t) for t in T) for r in R])
    ref=[r for r in R if r['idx']==0][0]
    pk[kn]=dict(n=len(R), J_mean=100*J.mean(), J_med=100*np.median(J), J_min=100*J.min(), J_max=100*J.max(),
                J_ref=100*max(rate(ref,t) for t in T), lut_min=min(r['n_lut'] for r in R), lut_max=max(r['n_lut'] for r in R),
                ff_min=min(r['n_ff'] for r in R), ff_max=max(r['n_ff'] for r in R), lat_min=min(r['lat'] for r in R), lat_max=max(r['lat'] for r in R),
                cells_min=min(r['n_cell'] for r in R), cells_max=max(r['n_cell'] for r in R),
                S=100*np.mean([rate(r,'1.2_1',0) for r in R]), D=100*np.mean([rate(r,'1.2_1',1) for r in R]), H=100*np.mean([rate(r,'1.2_1',3) for r in R]))
    print(kn, {k: round(v,3) if isinstance(v,float) else v for k,v in pk[kn].items()})
out['per_kernel']=pk
# 4. hamming vs CER twin
print('\nSpearman Hamming distance vs CER (twin r1.2) by floorplan')
hd={}
for fp in ['none','split','g1','g2']:
    S=[r for r in rows if r['fp']==fp]
    hd[fp]=spearmanr([ham(r) for r in S],[rate(r,'1.2_2') for r in S]).correlation
print({k: round(v,3) for k,v in hd.items()})
out['ham_rho']=hd
json.dump(out, open('analysis/e0.json','w'), indent=1, default=float)


### `evalpred.py` — Prediction metrics

In [ ]:
%%writefile evalpred.py
import sys, glob, json, numpy as np
from scipy.stats import spearmanr
THR = ['0.6_1','1.2_1','2.0_1','0.6_2','1.2_2','2.0_2']
def metrics(f):
    z = np.load(f, allow_pickle=True)
    P, R, CM, AN, CMI, ker = z['P'], z['R'], z['CM'], z['AN'], z['CMI'], z['ker']
    out = {}
    for k in np.unique(ker):
        m = ker == k
        pc, rc = P[m][:, :, 2], R[m][:, :, 2]
        J_p, J_r = pc.max(1), rc.max(1)
        rho_t = [spearmanr(pc[:, t], rc[:, t]).correlation for t in range(6)]
        n = 4096
        nll = -(R[m] * n * np.log(P[m] + 1e-6)).sum(-1).mean() / n
        cmi_p = CM[m] / np.maximum(AN[m], 1e-6) if AN[m].max() > 0 else CM[m]
        k10 = max(1, int(0.1 * m.sum()))
        top_true = set(np.argsort(J_r)[:k10]); top_pred = set(np.argsort(J_p)[:k10])
        out[k] = dict(mae_cer=100 * np.abs(pc - rc).mean(), mae_all=100 * np.abs(P[m] - R[m]).mean(),
                      rho_cer=np.nanmean(rho_t), rho_J=spearmanr(J_p, J_r).correlation, nll=nll,
                      cmi_mae=np.abs(np.clip(cmi_p, 0, 1) - CMI[m]).mean(), top10=len(top_true & top_pred) / k10,
                      rho_single=np.nanmean(rho_t[:3]), rho_twin=np.nanmean(rho_t[3:]))
    return out
if __name__ == '__main__':
    for f in sorted(glob.glob(sys.argv[1] if len(sys.argv) > 1 else 'results/pred_*.npz')):
        for k, v in metrics(f).items():
            print(f.split('/')[-1], k, {a: round(float(b), 3) for a, b in v.items()})


### `stats.py` — Kernel-level statistics for prediction

In [ ]:
%%writefile stats.py
"""Aggregate prediction results: per-model means/SD over held-out kernels, paired Wilcoxon (kernel-seed pairs), Holm, Cliff's delta."""
import sys, glob, json, collections, numpy as np
from evalpred import metrics
from scipy.stats import wilcoxon
KEYS = ['rho_J', 'rho_cer', 'rho_single', 'rho_twin', 'mae_cer', 'top10', 'nll', 'cmi_mae']
def load_all():
    agg = collections.defaultdict(dict)
    for f in sorted(glob.glob('results/pred_*.npz')):
        name = f.split('pred_')[1].rsplit('_f', 1)[0]
        seed = int(f.split('_s')[-1][:-4])
        z = np.load(f)
        for k, v in metrics(f).items():
            agg[name][(k, seed)] = dict(v, ttrain=float(z['ttrain']), tinf=float(z['tinf']), nparam=int(z['nparam']))
    return agg
def cliff(x, y):
    x, y = np.asarray(x), np.asarray(y)
    return ((x[:, None] > y[None]).sum() - (x[:, None] < y[None]).sum()) / (len(x) * len(y))
def holm(ps):
    order = np.argsort(ps); m = len(ps); adj = np.zeros(m); run = 0
    for r, i in enumerate(order):
        run = max(run, (m - r) * ps[i]); adj[i] = min(1, run)
    return adj
if __name__ == '__main__':
    agg = load_all()
    out = {}
    for name, d in agg.items():
        # average over seeds per kernel first, then mean/sd over kernels
        kern = collections.defaultdict(list)
        for (k, s), v in d.items():
            kern[k].append(v)
        per_k = {k: {key: float(np.mean([v[key] for v in L])) for key in KEYS} for k, L in kern.items()}
        out[name] = {key: (float(np.mean([p[key] for p in per_k.values()])), float(np.std([p[key] for p in per_k.values()]))) for key in KEYS}
        out[name]['per_kernel'] = per_k
        out[name]['n_runs'] = len(d)
        out[name]['ttrain'] = float(np.mean([v['ttrain'] for v in d.values()]))
        out[name]['tinf_ms'] = float(1000 * np.mean([v['tinf'] for v in d.values()]))
        out[name]['nparam'] = int(np.mean([v['nparam'] for v in d.values()]))
    # paired tests vs asunder on kernel-level means (seeds averaged), n = 8 held-out kernels
    tests = {}
    names = [n for n in out if n != 'asunder' and not n.startswith('_')]
    raw = []
    for metric in ['rho_J', 'rho_single', 'rho_twin']:
        for n in names:
            ks = sorted(set(out['asunder']['per_kernel']) & set(out[n]['per_kernel']))
            xs = [out['asunder']['per_kernel'][k][metric] for k in ks]; ys = [out[n]['per_kernel'][k][metric] for k in ks]
            p = wilcoxon(xs, ys).pvalue
            raw.append((metric, n, p, float(np.mean(np.array(xs) - np.array(ys))), float(cliff(xs, ys)), int(sum(x > y for x, y in zip(xs, ys))), len(xs)))
    for metric in ['rho_J', 'rho_single', 'rho_twin']:
        sub = [r for r in raw if r[0] == metric]
        adj = holm([r[2] for r in sub])
        for (mt, n, p, d, cd, wins, nn), pa in zip(sub, adj):
            tests[f'{mt}|{n}'] = dict(p=float(p), p_holm=float(pa), mean_diff=d, cliff=cd, wins=wins, n=nn)
    out['_tests_rhoJ'] = tests
    json.dump(out, open('analysis/pred_stats.json', 'w'), indent=1)
    for n in sorted(out, key=lambda n: -out[n]['rho_J'][0] if not n.startswith('_') else 0):
        if n.startswith('_'): continue
        print(f"{n:13s} " + ' '.join(f"{k}={out[n][k][0]:.3f}±{out[n][k][1]:.3f}" for k in ['rho_J', 'rho_single', 'rho_twin', 'mae_cer', 'top10', 'nll']), out[n]['n_runs'])
    for n, t in tests.items():
        print('test', n, {k: round(v, 4) if isinstance(v, float) else v for k, v in t.items()})


### `ablation.py` — Ablation summary

In [ ]:
%%writefile ablation.py
"""Ablation summary: every variant against the seed-0 full model on the same folds, plus the seed spread of the full model."""
import sys, json, os, numpy as np
from evalpred import metrics
from train import KERNEL_ORDER

KEYS = ['rho_J', 'rho_single', 'rho_twin', 'top10', 'mae_cer', 'nll']
VARIANTS = ['asunder', 'no_corr', 'no_prox', 'no_pos', 'pool_readout']
import glob


def per_kernel(name, seed):
    out = {}
    for f in range(4):
        fn = f'results/pred_{name}_f{f}_s{seed}.npz'
        if os.path.exists(fn):
            out.update(metrics(fn))
    return out


res = {}
for v in VARIANTS:
    pk = per_kernel(v, 0)
    if len(pk) < 8:
        continue
    res[v] = {k: [float(np.mean([pk[kn][k] for kn in KERNEL_ORDER])), float(np.std([pk[kn][k] for kn in KERNEL_ORDER]))] for k in KEYS}
    res[v]['per_kernel_rhoJ'] = {kn: float(pk[kn]['rho_J']) for kn in KERNEL_ORDER}
full = res['asunder']
for v in res:
    if v == 'asunder':
        continue
    d = [res[v]['per_kernel_rhoJ'][kn] - full['per_kernel_rhoJ'][kn] for kn in KERNEL_ORDER]
    res[v]['delta_rhoJ'] = float(np.mean(d))
    res[v]['kernels_worse'] = int(sum(x < 0 for x in d))
# seed-to-seed spread of the full model (mean over kernels of each seed, then SD over seeds)
seed_means = {k: [] for k in KEYS}
for s in range(3):
    pk = per_kernel('asunder', s)
    if len(pk) == 8:
        for k in KEYS:
            seed_means[k].append(float(np.mean([pk[kn][k] for kn in KERNEL_ORDER])))
res['_seed_sd'] = {k: float(np.std(v, ddof=1)) if len(v) > 1 else None for k, v in seed_means.items()}
res['_seed_range'] = {k: [min(v), max(v)] if v else None for k, v in seed_means.items()}
Ps = [per_kernel('asunder', s) for s in range(3)]
if all(len(p) == 8 for p in Ps):
    res['_seed_sd_kernel'] = {kn: float(np.std([p[kn]['rho_J'] for p in Ps], ddof=1)) for kn in KERNEL_ORDER}
json.dump(res, open('analysis/ablation.json', 'w'), indent=1)
for v in VARIANTS:
    if v in res:
        print(v, {k: round(res[v][k][0], 3) for k in KEYS}, res[v].get('delta_rhoJ'), res[v].get('kernels_worse'))
print('seed sd', res['_seed_sd'])
print('seed range', res['_seed_range'])


### `riskmap.py` — Per-position risk maps (localisation check)

In [ ]:
%%writefile riskmap.py
import sys, json
import numpy as np, torch
from dmr import Design
from place import place
from fi import cell_to_elem, classify
from netlist import lanes_to_words
from graph import build_graph
from train import to_torch, make_model
from exp_pred import SPECS
from scipy.stats import spearmanr
torch.set_num_threads(1)
REF = dict(u=1, alloc=1, share='le', fsm='bin', sched='asap')
DIV = dict(u=2, alloc=1, share='ded', fsm='onehot', sched='alap')
def grid_spots(pl, m, nx, ny):
    off = pl['offB']; W, H = pl['W'], pl['H']
    span = (W - off) if m == 2 else W
    xs = -0.5 + (np.arange(nx) + 0.5) / nx * span
    ys = -0.5 + (np.arange(ny) + 0.5) / ny * H
    C = np.array([(x, y) for y in ys for x in xs])
    return C, xs, ys
def empirical(d, pl, C, r, m, lanes=512, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    pos = pl['pos']
    for i, c in enumerate(C):
        inp = d.make_inputs(lanes, 100 + i)
        t = rng.integers(1, d.lat, size=lanes)
        cs = [c] if m == 1 else [c, c + np.array([pl['offB'], 0])]
        hit = np.zeros(len(pos), bool)
        for cc in cs:
            hit |= ((pos - cc) ** 2).sum(1) <= r * r
        H = np.repeat(hit[:, None], lanes, 1)
        ffh, luth = cell_to_elem(d, pl, H)
        res = d.run(inp[0], lanes_to_words(ffh), lanes_to_words(luth), t)
        o, cm, anyw = classify(d, res, inp[1])
        out.append(o[2].mean())
    return np.array(out)
@torch.no_grad()
def predicted(g, C, r, m, fold=3, seeds=(0,)):
    gt = to_torch(g)
    sp = torch.tensor(np.stack([C, C + (np.array([g['offB'], 0]) if m == 2 else 0)], 1), dtype=torch.float32)
    res = []
    for s in seeds:
        mdl = make_model(SPECS['asunder']); mdl.load_state_dict(torch.load(f'models/asunder_f{fold}_s{s}.pt')); mdl.eval()
        h = mdl.encode(gt)
        _, _, _, per = mdl.readout(gt, h, [(r, m)], [sp])
        res.append(per[0][:, 2].numpy())
    return np.mean(res, 0)
if __name__ == '__main__':
    kn = 'speck'; r = 1.2
    out = {}
    for name, a, b, fp in [('ident_g2', REF, REF, 'g2'), ('div_g2', REF, DIV, 'g2'), ('ident_none', REF, REF, 'none')]:
        d = Design(kn, a, b); pl = place(d.nl, fp, 0); g = build_graph(d, pl)
        for m in (1, 2):
            C, xs, ys = grid_spots(pl, m, 12 if m == 1 else 6, 10)
            e = empirical(d, pl, C, r, m)
            p = predicted(g, C, r, m)
            rho = spearmanr(e, p).correlation if e.std() > 0 else float('nan')
            out[f'{name}_m{m}'] = dict(C=C.tolist(), emp=e.tolist(), pred=p.tolist(), xs=xs.tolist(), ys=ys.tolist(),
                                     W=pl['W'], H=pl['H'], off=pl['offB'], rho=rho, emp_mean=float(e.mean()), pred_mean=float(p.mean()))
            print(name, m, 'rho', rho, 'emp mean', e.mean(), 'pred mean', p.mean(), flush=True)
    json.dump(out, open('analysis/riskmap.json', 'w'))


### `transfer.py` — Generalisation to unseen threats

In [ ]:
%%writefile transfer.py
import sys, json, os, glob
import numpy as np, torch
from scipy.stats import spearmanr
from train import load, FOLDS, KERNEL_ORDER, THREATS, to_torch, make_model, truth
from model import collate
from exp_pred import SPECS
from gbm import features, logit
from sklearn.ensemble import HistGradientBoostingRegressor
torch.set_num_threads(1)
NEW = [(0.9, 1), (0.9, 2), (2.6, 1), (2.6, 2), (1.2, 3)]

def spots(g, t, G, rng):
    W, H, off = g['W'], g['H'], g['offB']; r, m = t
    s = int(np.sqrt(G))
    if m == 2:
        span = W - off
    else:
        span = W
    u = (np.arange(s)[:, None] + rng.uniform(0, 1, (s, s))) / s
    v = (np.arange(s)[None, :] + rng.uniform(0, 1, (s, s))) / s
    cx = -0.5 + u.ravel() * span; cy = -0.5 + v.ravel() * H
    c1 = np.stack([cx, cy], 1)
    if m == 1:
        c2 = c1
    elif m == 2:
        c2 = c1 + np.array([off, 0])
    else:
        c2 = np.stack([rng.uniform(-0.5, W - 0.5, len(cx)), rng.uniform(-0.5, H - 0.5, len(cx))], 1)
    return torch.tensor(np.stack([c1, c2], 1), dtype=torch.float32)

@torch.no_grad()
def asunder_pred(fold, seed, gs):
    m = make_model(SPECS['asunder']); m.load_state_dict(torch.load(f'models/asunder_f{fold}_s{seed}.pt')); m.eval()
    rng = np.random.default_rng(5)
    P = np.zeros((len(gs), len(NEW)))
    for b0 in range(0, len(gs), 16):
        gl = [to_torch(g) for g in gs[b0:b0 + 16]]; B = collate(gl); H = m.encode(B)
        for j, g in enumerate(gl):
            h = H[B['ptr'][j]:B['ptr'][j + 1]]
            thr = [(r, min(mm, 2)) for r, mm in NEW]
            p, _, _, _ = m.readout(g, h, thr, [spots(g, t, 144, rng) for t in NEW])
            P[b0 + j] = p[:, 2].numpy()
    return P

def gbm_pred(fold, gs, mode):
    rng = np.random.default_rng(1000 + fold); tr = []
    for k in KERNEL_ORDER:
        if k in FOLDS[fold]: continue
        g2 = load([k]); idx = rng.permutation(len(g2))[:180]; tr += [g2[i] for i in idx]
    R, _ = truth(tr)
    X = np.array([features(g, t) for g in tr for t in THREATS])
    mdl = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=0).fit(X, logit(R[:, :, 2].reshape(-1)))
    P = np.zeros((len(gs), len(NEW)))
    for j, (r, mm) in enumerate(NEW):
        mf = mm if mm < 3 else mode
        Xp = np.array([features(g, (r, mf)) for g in gs])
        P[:, j] = 1 / (1 + np.exp(-mdl.predict(Xp)))
    return P

if __name__ == '__main__':
    res = {}
    for fold, held in enumerate(FOLDS):
        for kn in held:
            lab = json.load(open(f'data/transfer_{kn}.json'))
            gs = [g for g in load([kn]) if str(g['meta']['idx']) in lab]
            Y = np.array([[lab[str(g['meta']['idx'])][f'{r}_{m}']['counts'][2] / lab[str(g['meta']['idx'])][f'{r}_{m}']['n'] for r, m in NEW] for g in gs])
            preds = {f'asunder_s{s}': asunder_pred(fold, s, gs) for s in range(3) if os.path.exists(f'models/asunder_f{fold}_s{s}.pt')}
            preds['gbm_m1'] = gbm_pred(fold, gs, 1); preds['gbm_m2'] = gbm_pred(fold, gs, 2)
            res[kn] = {name: {f'{r}_{m}': dict(rho=float(spearmanr(P[:, j], Y[:, j]).correlation), mae=float(100 * np.abs(P[:, j] - Y[:, j]).mean()))
                              for j, (r, m) in enumerate(NEW)} for name, P in preds.items()}
            print(kn, {n: {t: round(v['rho'], 2) for t, v in d.items()} for n, d in res[kn].items()}, flush=True)
    json.dump(res, open('analysis/transfer.json', 'w'), indent=1)


### `search_stats.py` — Search statistics

In [ ]:
%%writefile search_stats.py
import json, glob, collections, numpy as np
from scipy.stats import wilcoxon
METHODS = ['random', 'diversity', 'gp_bo', 'reinforce', 'gbm_screen', 'asunder_screen', 'asunder', 'asunder_gp']
def cliff(x, y):
    x, y = np.asarray(x), np.asarray(y)
    return ((x[:, None] > y[None]).sum() - (x[:, None] < y[None]).sum()) / (len(x) * len(y))
def holm(ps):
    order = np.argsort(ps); m = len(ps); adj = np.zeros(m); run = 0
    for r, i in enumerate(order):
        run = max(run, (m - r) * ps[i]); adj[i] = min(1, run)
    return adj
res = {k.split('res_')[1][:-5]: json.load(open(k)) for k in glob.glob('search/res_*.json')}
J = collections.defaultdict(dict); REG = collections.defaultdict(dict); PCT = collections.defaultdict(dict); CURVE = collections.defaultdict(list)
OK = collections.defaultdict(dict); AREA = collections.defaultdict(dict); LAT = collections.defaultdict(dict)
for kn, r in res.items():
    pool_J = np.array(list(r['J_cache'].values()))
    Jmin, Jref = r['oracle_seen'], r['J_ref']
    for run in r['runs']:
        key = (kn, run['seed'])
        J[run['method']][key] = run['J_final']
        REG[run['method']][key] = (run['J_final'] - Jmin) / max(Jref - Jmin, 1e-9)
        PCT[run['method']][key] = float((pool_J < run['obs'][int(np.argmin(run['obs']))]).mean())
        c = list(run['curve']) + [run['curve'][-1]] * (16 - len(run['curve']))
        CURVE[run['method']].append([(x - Jmin) / max(Jref - Jmin, 1e-9) for x in c[:16]])
        AREA[run['method']][key] = run['area'] / r['ref_area']; LAT[run['method']][key] = run['lat'] / r['ref_lat']
out = {}
for m in METHODS:
    if m not in J: continue
    v = np.array(list(J[m].values())) * 100
    out[m] = dict(J_mean=float(v.mean()), J_median=float(np.median(v)), J_sd=float(v.std()), regret_mean=float(np.mean(list(REG[m].values()))),
                  regret_median=float(np.median(list(REG[m].values()))), pct_mean=float(np.mean(list(PCT[m].values()))),
                  area=float(np.mean(list(AREA[m].values()))), lat=float(np.mean(list(LAT[m].values()))),
                  curve=np.mean(CURVE[m], 0).tolist(), n=len(v))
kernels = sorted({k for k, _ in J['asunder']})
km = {m: np.array([np.mean([J[m][(kn, sd)] for sd in range(5) if (kn, sd) in J[m]]) for kn in kernels]) for m in METHODS if m in J}
tests = []
for m in METHODS:
    if m == 'asunder' or m not in km: continue
    a, b = km['asunder'], km[m]
    diff = a - b
    p = wilcoxon(a, b).pvalue if np.any(diff != 0) else 1.0
    tests.append((m, p, float(cliff(b, a)), int((diff < 0).sum()), int((diff > 0).sum()), int((diff == 0).sum())))
adj = holm([t[1] for t in tests])
out['_tests'] = {t[0]: dict(p=float(t[1]), p_holm=float(pa), cliff_other_vs_asunder=t[2], asunder_better=t[3], asunder_worse=t[4], ties=t[5]) for t, pa in zip(tests, adj)}
out['_per_kernel'] = {kn: dict(J_ref=100 * r['J_ref'], J_min=100 * r['oracle_seen'], pool=r['pool_size'], t_fi=r['t_fi'], t_surr=r['t_surrogate'],
                               **{m: 100 * float(np.mean([J[m][(kn, s)] for s in range(5) if (kn, s) in J[m]])) for m in METHODS if m in J}) for kn, r in res.items()}
json.dump(out, open('analysis/search_stats.json', 'w'), indent=1)
for m in METHODS:
    if m in out: print(f"{m:15s} J={out[m]['J_mean']:.3f}% med={out[m]['J_median']:.3f} regret={out[m]['regret_mean']:.3f} pct={out[m]['pct_mean']:.3f} area={out[m]['area']:.2f} lat={out[m]['lat']:.2f}")
for k, v in out['_tests'].items(): print(k, v)
for k, v in out['_per_kernel'].items(): print(k, {a: round(b, 3) if isinstance(b, float) else b for a, b in v.items()})


### `fig_arch.py` — Figure 1: framework

In [ ]:
%%writefile fig_arch.py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
plt.rcParams.update({'font.family':'serif','font.serif':['DejaVu Serif'],'mathtext.fontset':'dejavuserif'})
fig, ax = plt.subplots(figsize=(7.16, 4.1))
ax.set_xlim(-4, 101); ax.set_ylim(-1, 57); ax.axis('off')
W, H = 17.0, 15.0
def box(x, y, title, lines, fill='white'):
    ax.add_patch(FancyBboxPatch((x, y), W, H, boxstyle='round,pad=0,rounding_size=1.0', fc=fill, ec='black', lw=0.9))
    ax.text(x + W/2, y + H - 1.0, title, ha='center', va='top', fontsize=6.5, fontweight='bold', linespacing=1.1)
    ax.text(x + W/2, y + 1.2, '\n'.join(lines), ha='center', va='bottom', fontsize=6.2, linespacing=1.3)
def arr(pts, ls='-', label=None, lpos=None, ha='center'):
    for (x0, y0), (x1, y1) in zip(pts[:-2], pts[1:-1]):
        ax.plot([x0, x1], [y0, y1], color='black', lw=0.9, ls=ls)
    ax.add_patch(FancyArrowPatch(pts[-2], pts[-1], arrowstyle='-|>', mutation_scale=8, lw=0.9, ls=ls,
                                 color='black', shrinkA=0, shrinkB=0))
    if label:
        ax.text(lpos[0], lpos[1], label, fontsize=6.3, ha=ha, va='center', style='italic',
                bbox=dict(fc='white', ec='none', pad=0.4))
X = [1, 21.5, 42, 62.5, 83]
YT, YB = 34, 8
box(X[0], YT, 'Directive pair', [r'replica A: $\theta_A$', r'replica B: $\theta_B$', r'floorplan: $\varphi$'], '#eeeeee')
box(X[1], YT, 'HLS + synthesis', ['schedule, bind,', 'FSM emission,', 'LUT4/FF netlists', 'A, B and W'])
box(X[2], YT, 'Placement', ['quadratic solve,', 'bisection', 'legaliser, guard', r'band from $\varphi$'])
box(X[3], YT, 'Cross-replica\ngraph', ['netlist edges,', 'proximity edges,', 'correspondence', 'edges'])
box(X[4], YT, 'Typed-attention\nencoder', [r'$L$ layers,', 'one projection', 'set per edge', 'type'])
box(X[0], YB, 'Training loss', ['multinomial NLL,', 'binomial terms,', 'relative critical', r'deviance $\mathcal{L}_C$'], '#eeeeee')
box(X[1], YB, 'Spot-fault\ncampaigns', ['single and twin', r'spots of radius $r$,', 'outcome counts', 'and CMI'])
box(X[2], YB, 'Budgeted search', ['ensemble LCB,', 'few-shot', 'recalibration,', 'area/latency caps'])
box(X[3], YB, 'Outcome law', [r'$\hat{\pi}_S,\hat{\pi}_D,\hat{\pi}_C,\hat{\pi}_H$', r'and $\widehat{\mathrm{CMI}}$', r'per threat $\tau$'])
box(X[4], YB, 'Spot-integral\nreadout', [r'soft hit weights', r'$\omega_v(\mathbf{c};r)$, owner', 'pooling, mean', 'over spots'])
ym = YT + H/2
for i in range(4):
    arr([(X[i] + W, ym), (X[i + 1], ym)])
# encoder -> readout
arr([(X[4] + W/2, YT), (X[4] + W/2, YB + H)], label=r'$\mathbf{h}_v^{(L)}$', lpos=(X[4] + W/2 + 4.2, 28.5))
yb = YB + H/2
arr([(X[4], yb), (X[3] + W, yb)])
arr([(X[3], yb), (X[2] + W, yb)])
arr([(X[2], yb), (X[1] + W, yb)])
arr([(X[1], yb), (X[0] + W, yb)])
# search -> directive pair (proposals) through the gap between rows
xs = X[2] + W/2
arr([(xs, YB + H), (xs, 28.0), (X[0] + W/2, 28.0), (X[0] + W/2, YT)],
    label=r'proposals $(\theta_A,\theta_B,\varphi)$', lpos=(26.5, 28.0))
# outcome law -> loss, routed below the bottom row
xo = X[3] + W/2
arr([(xo, YB), (xo, 3.5), (X[0] + W/2, 3.5), (X[0] + W/2, YB)],
    label=r'predictions $\hat{\pi}$, $\widehat{\mathrm{CMI}}$', lpos=(40, 3.5))
# reverse-mode gradient, dashed, left margin then over the top row into the encoder
arr([(X[0], YB + H/2), (-2.5, YB + H/2), (-2.5, 53.5), (X[4] + W/2, 53.5), (X[4] + W/2, YT + H)], ls=(0, (4, 2.5)),
    label=r'reverse-mode gradient $\nabla_{\Theta}\mathcal{L}$ (training only)', lpos=(48, 53.5))
fig.savefig('figs/fig1_framework.png', dpi=600, bbox_inches='tight')


### `fig_threat.py` — Figure 2: placed designs and spots

In [ ]:
%%writefile fig_threat.py
import sys
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from dmr import Design
from place import place
from fi import campaign
plt.rcParams.update({'font.family':'serif','font.serif':['DejaVu Serif'],'font.size':7.5,'mathtext.fontset':'dejavuserif'})
REF = dict(u=1, alloc=1, share='le', fsm='bin', sched='asap')
DIV = dict(u=2, alloc=1, share='ded', fsm='onehot', sched='alap')
cases = [('(a) identical, no floorplan', REF, REF, 'none'), ('(b) identical, guard band g2', REF, REF, 'g2'),
         ('(c) diverse, guard band g2', REF, DIV, 'g2')]
fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.75))
out = []
for ax, (t, a, b, fp) in zip(axs, cases):
    d = Design('fir4', a, b); pl = place(d.nl, fp, 0)
    inp = d.make_inputs(8192, 5)
    c1 = campaign(d, pl, 1.2, 1, 8192, 11, inp)['rates'][2]
    c2 = campaign(d, pl, 1.2, 2, 8192, 12, inp)['rates'][2]
    out.append((t, c1, c2))
    pos, own = pl['pos'], pl['owner']
    for o, mk, fc, lab in [('A', 'o', 'black', 'replica A'), ('B', '^', '0.65', 'replica B'), ('W', 's', 'white', 'comparator W')]:
        m = own == o
        ax.scatter(pos[m, 0], pos[m, 1], s=4 if o != 'W' else 7, marker=mk, c=fc, edgecolors='black' if o == 'W' else 'none',
                   linewidths=0.4, label=lab, zorder=2)
    cx, cy = 1.2 + 0.1 * pl['W'], pl['H'] * 0.55
    ax.add_patch(Circle((cx, cy), 1.2, fill=False, lw=1.0, ls='-', zorder=3))
    if fp != 'none':
        ax.add_patch(Circle((cx + pl['offB'], cy), 1.2, fill=False, lw=1.0, ls='--', zorder=3))
        ax.annotate('', xy=(cx + pl['offB'] - 1.25, cy), xytext=(cx + 1.25, cy),
                    arrowprops=dict(arrowstyle='->', lw=0.7, ls=':'))
        ax.text(cx + pl['offB'] / 2 + 0.6, cy - 0.75, r'$\Delta$', ha='center', fontsize=7.5, bbox=dict(fc='white', ec='none', pad=0.3), zorder=4)
    ax.set_xlim(-1, pl['W']); ax.set_ylim(-1.2, pl['H'] + 0.2)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(t, fontsize=7.5, pad=3)
    ax.set_xlabel(f'CER single {100*c1:.2f}%   twin {100*c2:.2f}%', fontsize=7)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc='lower center', ncol=3, frameon=False, fontsize=7, markerscale=2.5, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.07, 1, 1), w_pad=1.2)
fig.savefig('figs/fig2_threat.png', dpi=600, bbox_inches='tight')
print(out)


### `fig_interaction.py` — Figure 3: floorplan-diversity interaction

In [ ]:
%%writefile fig_interaction.py
import sys, json
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from train import KERNEL_ORDER
plt.rcParams.update({'font.family':'serif','font.serif':['DejaVu Serif'],'font.size':7.5,'mathtext.fontset':'dejavuserif'})
rows = [json.loads(l) for k in KERNEL_ORDER for l in open(f'data/{k}.jsonl')]
rate = lambda r, t: r['labels'][t]['counts'][2] / r['labels'][t]['n']
ham = lambda r: sum(r['cfgA'][f] != r['cfgB'][f] for f in r['cfgA'])
rng = np.random.default_rng(0)
def ci(v):
    b = [rng.choice(v, len(v)).mean() for _ in range(2000)]
    return np.mean(v), np.percentile(b, 2.5), np.percentile(b, 97.5)
FPS = ['none', 'split', 'g1', 'g2']
fig = plt.figure(figsize=(7.16, 2.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.15], wspace=0.42)
stats = {}
for j, (t, ttl) in enumerate([('1.2_1', '(a) single spot, $r=1.2$'), ('1.2_2', '(b) twin spot, $r=1.2$')]):
    ax = fig.add_subplot(gs[0, j])
    for k, (div, fc, hatch) in enumerate([('ident', '0.25', ''), ('diverse', 'white', '////')]):
        m, lo, hi = [], [], []
        for fp in FPS:
            v = np.array([100 * rate(r, t) for r in rows if r['fp'] == fp and (ham(r) == 0) == (div == 'ident')])
            a, b, c = ci(v); m.append(a); lo.append(a - b); hi.append(c - a)
            stats[f'{t}_{fp}_{div}'] = (a, b, c, len(v))
        x = np.arange(4) + (k - 0.5) * 0.36
        ax.bar(x, m, 0.34, yerr=[lo, hi], color=fc, edgecolor='black', hatch=hatch, lw=0.6,
               error_kw=dict(lw=0.6, capsize=1.8), label='identical replicas' if div == 'ident' else 'diverse replicas')
    ax.set_xticks(range(4)); ax.set_xticklabels(FPS)
    ax.set_ylabel('critical error rate (%)'); ax.set_title(ttl, fontsize=7.5)
    ax.spines[['top', 'right']].set_visible(False)
    if j == 0:
        ax.legend(frameon=False, fontsize=6.5, loc='upper right')
ax = fig.add_subplot(gs[0, 2])
e0 = json.load(open('analysis/e0.json'))['rho_dbf']
T = ['0.6_1', '1.2_1', '2.0_1', '0.6_2', '1.2_2', '2.0_2']
M = np.array([[e0[k][t] for t in T] for k in KERNEL_ORDER])
im = ax.imshow(M, cmap='Greys', vmin=-0.2, vmax=1.0, aspect='auto')
for i in range(M.shape[0]):
    for jj in range(M.shape[1]):
        ax.text(jj, i, f'{M[i, jj]:.2f}', ha='center', va='center', fontsize=5.6, color='white' if M[i, jj] > 0.55 else 'black')
ax.set_xticks(range(6)); ax.set_xticklabels([f'{t.split("_")[0]}\n{"S" if t.endswith("1") else "T"}' for t in T], fontsize=6)
ax.set_yticks(range(8)); ax.set_yticklabels(KERNEL_ORDER, fontsize=6)
ax.set_title(r'(c) Spearman $\rho$, blind DBF vs spot', fontsize=7.5)
cb = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.03); cb.ax.tick_params(labelsize=6)
fig.savefig('figs/fig3_interaction.png', dpi=600, bbox_inches='tight')
json.dump({k: list(map(float, v)) for k, v in stats.items()}, open('analysis/fig3_stats.json', 'w'), indent=1)


### `fig_results.py` — Figures 4 and 5: prediction and search

In [ ]:
%%writefile fig_results.py
import sys, json, glob
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from train import KERNEL_ORDER
plt.rcParams.update({'font.family':'serif','font.serif':['DejaVu Serif'],'font.size':7.5,'mathtext.fontset':'dejavuserif'})
ps = json.load(open('analysis/pred_stats.json'))
# ---------------- Fig 4: prediction per held-out kernel + predicted vs measured scatter
models = [('asunder', 'ASUNDER', '0.15', ''), ('gbm', 'GBM + exposure', '0.55', ''), ('gbm_noexp', 'GBM, no exposure', 'white', '////'),
          ('gcn', 'GCN', 'white', '....'), ('gat', 'GAT', 'white', 'xxxx'), ('gin', 'GIN', 'white', '\\\\\\\\')]
models = [m for m in models if m[0] in ps]
fig = plt.figure(figsize=(7.16, 2.7))
gs = fig.add_gridspec(1, 2, width_ratios=[2.3, 1], wspace=0.28)
ax = fig.add_subplot(gs[0, 0])
w = 0.8 / len(models)
for i, (key, lab, fc, hatch) in enumerate(models):
    v = [ps[key]['per_kernel'][k]['rho_J'] for k in KERNEL_ORDER]
    ax.bar(np.arange(8) + (i - (len(models) - 1) / 2) * w, v, w * 0.95, color=fc, edgecolor='black', hatch=hatch, lw=0.5, label=lab)
ax.set_xticks(range(8)); ax.set_xticklabels(KERNEL_ORDER, fontsize=6.6, rotation=20, ha='right')
ax.set_ylabel(r'Spearman $\rho$ of $J$ (held-out kernel)'); ax.set_ylim(min(0, ax.get_ylim()[0]), 1.0)
ax.axhline(0, color='black', lw=0.5)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(ncol=3, frameon=False, fontsize=6.3, loc='upper left', bbox_to_anchor=(0.0, 1.17))
ax.set_title('(a)', fontsize=7.5, loc='left', x=-0.08, y=1.08)
ax = fig.add_subplot(gs[0, 1])
xs, ys = [], []
for f in glob.glob('results/pred_asunder_f*_s0.npz'):
    z = np.load(f)
    xs += list(z['R'][:, :, 2].max(1)); ys += list(z['P'][:, :, 2].max(1))
xs, ys = 100 * np.array(xs) + 0.01, 100 * np.array(ys) + 0.01
ax.scatter(xs, ys, s=2.5, c='black', alpha=0.35, lw=0)
lim = [0.008, 60]
ax.plot(lim, lim, 'k--', lw=0.6)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(r'measured $J$ (%)'); ax.set_ylabel(r'predicted $J$ (%)')
ax.set_title('(b)', fontsize=7.5, loc='left', x=-0.25, y=1.08)
ax.spines[['top', 'right']].set_visible(False)
fig.savefig('figs/fig4_prediction.png', dpi=600, bbox_inches='tight')
# ---------------- Fig 5: search
ss = json.load(open('analysis/search_stats.json'))
meth = [('asunder', 'ASUNDER', '-', 'o', 'black'), ('gp_bo', 'GP-BO', '--', 's', '0.35'), ('reinforce', 'REINFORCE', '-.', '^', '0.35'),
        ('random', 'random', ':', 'v', '0.45'), ('asunder_screen', 'ASUNDER screen', '-', 'D', '0.65'), ('gbm_screen', 'GBM screen', '--', 'x', '0.65')]
fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.6), gridspec_kw=dict(width_ratios=[1, 1.35], wspace=0.3))
ax = axs[0]
for key, lab, ls, mk, c in meth:
    cv = ss[key]['curve']
    ax.plot(np.arange(1, len(cv) + 1), cv, ls=ls, marker=mk, ms=2.6, color=c, lw=0.9, label=lab, markevery=3)
ax.set_yscale('log'); ax.set_xlabel('fault-injection campaigns used'); ax.set_ylabel('normalised regret of best found')
ax.legend(frameon=False, fontsize=6.2); ax.spines[['top', 'right']].set_visible(False)
ax.set_title('(a)', fontsize=7.5, loc='left', x=-0.2)
ax = axs[1]
pk = ss['_per_kernel']
order = ['asunder', 'gp_bo', 'reinforce', 'random', 'asunder_screen', 'gbm_screen', 'asunder_gp', 'diversity']
labels = ['ASUNDER', 'GP-BO', 'REINFORCE', 'random', 'ASUNDER screen', 'GBM screen', 'ASUNDER-GP', 'diversity']
mk = dict(zip(KERNEL_ORDER, ['o', 's', '^', 'v', 'D', 'P', 'X', '*']))
for i, k in enumerate(KERNEL_ORDER):
    if k not in pk: continue
    vals = [pk[k][m] + 0.005 for m in order]
    ax.plot(np.arange(len(order)) + (i - 3.5) * 0.075, vals, ls='none', marker=mk[k], ms=3.2, mfc='white' if i % 2 else 'black', mec='black', mew=0.6, label=k)
ax.set_yscale('log'); ax.set_xticks(range(len(order))); ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=6.5)
ax.set_ylabel(r're-measured $J$ of returned design (%)')
ax.legend(ncol=4, frameon=False, fontsize=5.8, loc='lower center', bbox_to_anchor=(0.5, 1.0), handletextpad=0.2, columnspacing=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.set_title('(b)', fontsize=7.5, loc='left', x=-0.12, y=1.0)
fig.savefig('figs/fig5_search.png', dpi=600, bbox_inches='tight')
print('ok')


### `fig_threats_ablation.py` — Per-threat ranking, ablation and risk-map figures

In [ ]:
%%writefile fig_threats_ablation.py
"""Fig. 6: ranking quality per threat (training and unseen threats). Fig. 7: ablation per kernel. Fig. 8: per-position risk maps."""
import sys, json, os, glob
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from train import KERNEL_ORDER
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif'], 'font.size': 7.5, 'mathtext.fontset': 'dejavuserif',
                     'axes.spines.top': False, 'axes.spines.right': False})


def per_threat(name, seeds=(0, 1, 2)):
    """Mean over kernels of the seed-averaged Spearman correlation for each of the six training threats."""
    acc = {kn: [] for kn in KERNEL_ORDER}
    for s in seeds:
        for f in range(4):
            fn = f'results/pred_{name}_f{f}_s{s}.npz'
            if not os.path.exists(fn):
                continue
            z = np.load(fn, allow_pickle=True)
            P, R, ker = z['P'], z['R'], z['ker']
            for kn in np.unique(ker):
                m = ker == kn
                acc[str(kn)].append([spearmanr(P[m][:, t, 2], R[m][:, t, 2]).correlation for t in range(6)])
    per_k = np.array([np.nanmean(acc[kn], 0) for kn in KERNEL_ORDER if acc[kn]])
    return per_k.mean(0), per_k.std(0)


# ---------------------------------------------------------------- Fig. 6
models = [('asunder', 'ASUNDER', 'o', 'black', '-'), ('gbm', 'GBM + exposure', 's', '0.45', '--'),
          ('gbm_noexp', 'GBM, no exposure', 'D', '0.65', ':'), ('gcn', 'GCN', 'v', '0.3', '-.')]
train_lab = ['0.6\nsingle', '1.2\nsingle', '2.0\nsingle', '0.6\ntwin', '1.2\ntwin', '2.0\ntwin']
tr = json.load(open('analysis/transfer.json'))
new_keys = ['0.9_1', '2.6_1', '0.9_2', '2.6_2', '1.2_3']
new_lab = ['0.9\nsingle', '2.6\nsingle', '0.9\ntwin', '2.6\ntwin', '1.2\nrandom\ndouble']
fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.55), gridspec_kw={'width_ratios': [6, 5], 'wspace': 0.28})
ax = axs[0]
x = np.arange(6)
for i, (k, lab, mk, c, ls) in enumerate(models):
    mu, sd = per_threat(k)
    xx = x + (i - 1.5) * 0.09
    ax.errorbar(xx, mu, yerr=sd, fmt=mk, color=c, mfc=c if i % 2 == 0 else 'white', mec=c, ms=4, lw=0.8, capsize=1.5, label=lab)
    ax.plot(xx[:3], mu[:3], ls=ls, color=c, lw=0.7); ax.plot(xx[3:], mu[3:], ls=ls, color=c, lw=0.7)
ax.axvline(2.5, color='0.8', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(train_lab, fontsize=6.6)
ax.set_ylim(-0.1, 1.0); ax.set_ylabel('Spearman $\\rho_s$ of CER (held-out kernel)')
ax.set_xlabel('training threat: radius (tiles) and spot mode')
ax.set_title('(a)', fontsize=7.5, loc='left', x=-0.16, y=1.02)
ax.legend(ncol=2, frameon=False, fontsize=6.0, loc='lower center', bbox_to_anchor=(0.55, 1.03), handletextpad=0.3, columnspacing=1.2)
ax = axs[1]
x = np.arange(5)
series = [('ASUNDER', [np.mean([np.mean([tr[kn][f'asunder_s{s}'][t]['rho'] for s in range(3) if f'asunder_s{s}' in tr[kn]]) for kn in tr]) for t in new_keys],
           [np.std([np.mean([tr[kn][f'asunder_s{s}'][t]['rho'] for s in range(3) if f'asunder_s{s}' in tr[kn]]) for kn in tr]) for t in new_keys], 'o', 'black'),
          ('GBM, single-spot code', [np.mean([tr[kn]['gbm_m1'][t]['rho'] for kn in tr]) for t in new_keys],
           [np.std([tr[kn]['gbm_m1'][t]['rho'] for kn in tr]) for t in new_keys], 's', '0.45'),
          ('GBM, twin-spot code', [np.mean([tr[kn]['gbm_m2'][t]['rho'] for kn in tr]) for t in new_keys],
           [np.std([tr[kn]['gbm_m2'][t]['rho'] for kn in tr]) for t in new_keys], '^', '0.6')]
for i, (lab, mu, sd, mk, c) in enumerate(series):
    xx = x + (i - 1) * 0.12
    ax.errorbar(xx, mu, yerr=sd, fmt=mk, color=c, mfc=c if i != 2 else 'white', mec=c, ms=4, lw=0.8, capsize=1.5, label=lab)
ax.axvline(3.5, color='0.8', lw=0.6)
ax.set_xticks(x); ax.set_xticklabels(new_lab, fontsize=6.6)
ax.set_ylim(-0.1, 1.0)
ax.set_xlabel('unseen threat')
ax.set_title('(b)', fontsize=7.5, loc='left', x=-0.16, y=1.02)
ax.legend(ncol=2, frameon=False, fontsize=6.0, loc='lower center', bbox_to_anchor=(0.55, 1.03), handletextpad=0.3, columnspacing=1.2)
fig.savefig('figs/fig6_threats.png', dpi=600, bbox_inches='tight')

# ---------------------------------------------------------------- Fig. 7
ab = json.load(open('analysis/ablation.json'))
var = [('no_corr', 'no correspondence edges', 'o', 'black'), ('no_prox', 'no proximity edges', 's', '0.35'),
       ('no_pos', 'no encoder geometry', '^', '0.55'), ('pool_readout', 'pooled readout', 'D', '0.7')]
var = [v for v in var if v[0] in ab]
fig, ax = plt.subplots(figsize=(3.5, 2.5))
x = np.arange(8)
sdk = ab.get('_seed_sd_kernel', {})
for j, kn in enumerate(KERNEL_ORDER):
    if kn in sdk:
        ax.fill_between([j - 0.42, j + 0.42], -2 * sdk[kn], 2 * sdk[kn], color='0.9', lw=0)
ax.axhline(0, color='0.5', lw=0.6)
for i, (k, lab, mk, c) in enumerate(var):
    d = [ab[k]['per_kernel_rhoJ'][kn] - ab['asunder']['per_kernel_rhoJ'][kn] for kn in KERNEL_ORDER]
    ax.plot(x + (i - 1.5) * 0.15, d, mk, color=c, mfc=c if i % 2 == 0 else 'white', mec=c, ms=4, label=lab, lw=0)
ax.set_xticks(x); ax.set_xticklabels(KERNEL_ORDER, fontsize=6.4, rotation=25, ha='right')
ax.set_ylabel('$\\Delta\\rho_s$ of $J$ relative to full model')
ax.legend(ncol=2, frameon=False, fontsize=5.9, loc='lower center', bbox_to_anchor=(0.5, 1.0), handletextpad=0.2, columnspacing=0.7)
fig.savefig('figs/fig7_ablation.png', dpi=600, bbox_inches='tight')

# ---------------------------------------------------------------- Fig. 8
if not os.path.exists('analysis/riskmap.json'):
    print('ok (no risk map)'); sys.exit(0)
rm = json.load(open('analysis/riskmap.json'))['ident_g2_m2']
nx, ny = len(rm['xs']), len(rm['ys'])
E = 100 * np.array(rm['emp']).reshape(ny, nx); P = 100 * np.array(rm['pred']).reshape(ny, nx)
fig, axs = plt.subplots(1, 2, figsize=(3.5, 2.2), gridspec_kw={'wspace': 0.75})
for ax, M, t in [(axs[0], E, '(a) measured'), (axs[1], P, '(b) predicted')]:
    im = ax.imshow(M, origin='lower', cmap='Greys', aspect='auto',
                   extent=[rm['xs'][0] - 0.5 * (rm['xs'][1] - rm['xs'][0]), rm['xs'][-1] + 0.5 * (rm['xs'][1] - rm['xs'][0]),
                           rm['ys'][0] - 0.5 * (rm['ys'][1] - rm['ys'][0]), rm['ys'][-1] + 0.5 * (rm['ys'][1] - rm['ys'][0])])
    ax.set_title(t, fontsize=7.2, loc='left')
    ax.set_xlabel('spot $x$ (tiles)')
    if ax is axs[0]:
        ax.set_ylabel('spot $y$ (tiles)')
    cb = fig.colorbar(im, ax=ax, fraction=0.06, pad=0.04); cb.ax.tick_params(labelsize=6); cb.set_label('CER (%)', fontsize=6.5)
fig.savefig('figs/fig8_riskmap.png', dpi=600, bbox_inches='tight')
print('ok')


## 1. Verify the HLS flow bit-exactly against the golden models

In [ ]:
import random
from kernels import KERNELS
from hls import all_configs, cfg_key
from dmr import Design
for kn in KERNELS:
    cf = all_configs(KERNELS[kn]); rng = random.Random(3)
    a, b = rng.choice(cf), rng.choice(cf)
    ok, info = Design(kn, a, b).check()
    print(f'{kn:8s} {cfg_key(a)} | {cfg_key(b)} ->', 'OK' if ok else 'FAIL', info)

## 2. Synthesise every replica configuration (cached JSON netlists)

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from dmr import replica_json, wrapper_json
def synth_kernel(kn):
    wrapper_json(kn)
    for c in all_configs(KERNELS[kn]):
        replica_json(kn, c)
    return kn
with ProcessPoolExecutor(WORKERS) as ex:
    for kn in ex.map(synth_kernel, list(KERNELS)):
        print('synthesised', kn)

## 3. Fault-injection dataset (six spot threats + placement-blind DBF per design)

In [ ]:
import subprocess
ks = list(KERNELS)
groups = [','.join(ks[i::WORKERS]) for i in range(WORKERS)]
procs = [subprocess.Popen(['python', 'gen_data.py', g]) for g in groups if g]
[p.wait() for p in procs]
!wc -l data/*.jsonl

In [ ]:
!python cache_graphs.py aes_ark,aes_mix,present,speck,crc16,fir4,dct4,cordic
!mkdir -p analysis && python e0.py

## 4. Cross-kernel prediction: ASUNDER, ablations and baselines (leave-two-kernels-out folds)

In [ ]:
jobs = [f'{m}:{f}:{s}' for s in SEEDS for f in range(4) for m in ['asunder', 'gcn', 'gat', 'gin']]
jobs += [f'{m}:{f}:0' for f in range(4) for m in ['no_corr', 'no_prox', 'no_pos', 'pool_readout']]
chunks = [jobs[i::WORKERS] for i in range(WORKERS)]
procs = [subprocess.Popen(['python', 'exp_pred.py'] + c) for c in chunks if c]
[p.wait() for p in procs]
!python gbm.py

In [ ]:
import collections, numpy as np, json
!python stats.py
!python ablation.py
!python riskmap.py
ps = json.load(open('analysis/pred_stats.json'))
print(f"{'model':14s} {'rho_J':>7s} {'single':>7s} {'twin':>7s} {'MAE_C':>7s} {'top10':>6s}")
for name, v in ps.items():
    if name.startswith('_'):
        continue
    print(f"{name:14s} {v['rho_J'][0]:7.3f} {v['rho_single'][0]:7.3f} {v['rho_twin'][0]:7.3f} {v['mae_cer'][0]:7.3f} {v['top10'][0]:6.2f}")

## 5. Generalisation to threats never used in training (radius 0.9 and 2.6, random double spot)

In [ ]:
ks = list(KERNELS)
procs = [subprocess.Popen(['python', 'gen_transfer.py', ','.join(ks[i::WORKERS])]) for i in range(WORKERS) if ks[i::WORKERS]]
[p.wait() for p in procs]
!python transfer.py

## 6. Budgeted radius-robust search (requires the ASUNDER fold models for seeds 0..2; FAST mode uses the available seeds)

In [ ]:
import search
search_seeds = tuple(SEEDS)
_orig = search.ensemble_scores
search.ensemble_scores = lambda kn, graphs, seeds=search_seeds: _orig(kn, graphs, seeds)
os.makedirs('search', exist_ok=True)
for kn in KERNELS:
    search.run_kernel(kn)

In [ ]:
import search_gp
for kn in KERNELS:
    search_gp.run(kn)
!python search_stats.py
ss = json.load(open('analysis/search_stats.json'))
for m, v in sorted(((k, v) for k, v in ss.items() if not k.startswith('_')), key=lambda kv: kv[1]['J_mean']):
    print(f"{m:15s} mean J = {v['J_mean']:.3f}%  median = {v['J_median']:.3f}%  regret = {v['regret_mean']:.3f}")

## 7. Figures (600 dpi PNG in `figs/`)

In [ ]:
!python fig_arch.py && python fig_threat.py && python fig_interaction.py && python fig_results.py && python fig_threats_ablation.py
from IPython.display import Image, display
for f in ['fig1_framework', 'fig2_threat', 'fig3_interaction', 'fig4_prediction', 'fig5_search', 'fig6_threats', 'fig7_ablation', 'fig8_riskmap']:
    display(Image(f'figs/{f}.png', width=900))